In [1]:
# ============================================================
# 042_weekly_rq_status
# ============================================================
#
# Overview
# ----------------
# This notebook updates the weekly understanding of each Research Question (RQ)
# by:
#   1) Screening this week's evidence (papers/events) for RQ relevance
#   2) Proposing concrete, overwrite-ready updates to specific RQ fields
#   3) Writing the proposals as per-category weekly update pages in Notion
#      (one page per RQ × Category), with evidence linked via relations
#   4) Linking all created weekly update pages back to the most recent Weekly Digest
#
# The output is designed for fast review: each Notion page is scoped to ONE change
# category (e.g., Rationale, Approach, Gap), making it clear what is being updated
# and why, with citations and direct relations to evidence.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - WEEKLY_DIGESTS_DB: Most recent weekly digest record (immutable context; only "RQ Updates" relation is patched)
#   - RQ_DB: Target research questions (e.g., Priority=High) with current field texts
#   - PAPERS_DB: Papers ingested in the last N days with Decision="READ" or "KEEP"
#                (uses rich_text fields like Findings/Core Idea/Notes/Methods/Datasets for higher-quality proposals)
#   - EVENTS_DB: Weekly events within the digest week window
#   - WEEKLY_RQ_UPDATE_DB: Destination DB for weekly per-category RQ update pages
#
# Outputs:
#   - New per-category pages in WEEKLY_RQ_UPDATE_DB (one page per RQ × Category), including:
#     * Category (rich text): one of ["RQ","LinkedPaper","Rationale","Approach","Gap","NoChange"]
#     * Open Questions (rich text): Japanese UPDATED FINAL TEXT to store (overwrite-ready proposal)
#       - For "Rationale"/"Approach"/"Gap": the full updated field content (or gap bullets)
#       - For "RQ": the updated RQ wording/question
#       - For "LinkedPaper": Japanese explanation of papers to add and why
#     * Update Summary (rich text): reason/justification with explicit evidence citations (paper/event titles + P#/E#)
#     * Evidence Papers / Evidence Events (relations): concrete links to supporting items
#     * Confidence (number 0–1)
#     * Status (select): Proposed / Accepted / Rejected / Deferred
#     * Research Question (relation): link back to the RQ page
#     * Week (relation): link back to the Weekly Digest page
#   - Updated "RQ Updates" relation on the most recent WEEKLY_DIGESTS_DB page (patched; other fields unchanged)
#
# Structure
# ----------------
# Cell 01: Environment setup and imports
# Cell 02: Load configuration from env.txt
# Cell 03: Initialize Notion client, verify auth, and resolve database_id -> data_source_id
# Cell 04: Introspect WEEKLY_RQ_UPDATE_DB properties (schema-aware writes)
# Cell 05: Introspect RQ_DB properties (schema-aware reads)
# Cell 06: Introspect PAPERS_DB properties (schema-aware reads)
# Cell 07: Load most recent weekly digest record (context + week window)
# Cell 08: Load target research questions from RQ_DB (e.g., Priority=High)
# Cell 09: Load recent papers (Decision=READ/KEEP) as evidence candidates
# Cell 10: Load weekly events within the digest week window
# Cell 11: LLM screening + per-category proposals (quotes current RQ fields; outputs multiple items if needed)
# Cell 12: Normalize per-category items (Open Questions = proposed updated text; Update Summary = reason; confidence clamping)
# Cell 13: Build relation payloads for evidence linking (papers/events) for each item
# Cell 14: Write per-category pages to WEEKLY_RQ_UPDATE_DB and verify evidence relations
# Cell 15: Patch Weekly Digest "RQ Updates" relation to include created pages
# Cell 16: Generate execution summary and validation report
#
# Notes
# ----------------
# - Weekly Digest is treated as immutable context; only its "RQ Updates" relation is updated.
# - Evidence is filtered strictly: papers must be within the lookback window and Decision must be "READ" or "KEEP".
# - Notion querying uses data_sources/*/query in this environment; database queries may be invalid.
# - If database.properties is empty, schema is inferred from a sample page.
# - Confidence is meant to reflect evidence strength and clarity of the proposed update (not subjective certainty).
#


In [2]:
# ============================================================
# Cell 01 — Environment setup and imports
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# --- Mandatory env loading ---
from dotenv import load_dotenv
load_dotenv('env.txt')

# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

# --- Standard library imports ---
import os
import json
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional

# --- Third-party imports ---
import requests
from openai import OpenAI

# --- Initialize OpenAI client ---
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# --- Shared configuration ---
NOTION_VERSION = '2022-06-28'
DATE_FORMAT = '%Y-%m-%d'
DATETIME_FORMAT = '%Y-%m-%dT%H:%M:%S.%fZ'

# --- Helper function for safe property extraction ---
def get_notion_property(page: Dict, prop_name: str, prop_type: str) -> Any:
    """
    Safely extract a property value from a Notion page object.
    
    Args:
        page: Notion page dictionary
        prop_name: Name of the property
        prop_type: Type of property (title, rich_text, select, date, relation, number, etc.)
    
    Returns:
        Extracted value or None if not found/empty
    """
    try:
        prop = page.get('properties', {}).get(prop_name, {})
        
        if prop_type == 'title':
            return prop.get('title', [{}])[0].get('plain_text', '') if prop.get('title') else ''
        elif prop_type == 'rich_text':
            return prop.get('rich_text', [{}])[0].get('plain_text', '') if prop.get('rich_text') else ''
        elif prop_type == 'select':
            return prop.get('select', {}).get('name') if prop.get('select') else None
        elif prop_type == 'multi_select':
            return [item.get('name') for item in prop.get('multi_select', [])]
        elif prop_type == 'date':
            return prop.get('date', {}).get('start') if prop.get('date') else None
        elif prop_type == 'relation':
            return [item.get('id') for item in prop.get('relation', [])]
        elif prop_type == 'number':
            return prop.get('number')
        elif prop_type == 'checkbox':
            return prop.get('checkbox', False)
        else:
            return None
    except (KeyError, IndexError, TypeError):
        return None

print('Environment setup complete.')
print(f'LLM Provider: {llm_provider}')
print(f'LLM Model: {llm_model}')
print(f'LLM Temperature: {llm_temperature}')


Environment setup complete.
LLM Provider: OpenAI
LLM Model: gpt-4o-mini
LLM Temperature: 0.0


In [3]:
# ============================================================
# Cell 02 — Load configuration from env.txt
# ============================================================
# Overview:
#   Load environment variables and define shared configuration/constants used
#   throughout the notebook (IDs, time window, defaults). Keep this cell focused
#   on configuration only (no API calls).
#
# Inputs / Outputs:
#   Inputs:
#     - env.txt (loaded via load_dotenv)
#   Outputs:
#     - OPENAI_API_KEY, NOTION_TOKEN, NOTION_VERSION
#     - NOTION_*_DB_ID variables (data source IDs)
#     - LOOKBACK_DAYS, LOOKBACK_START (datetime), NOW (datetime)
#
# Notes:
#   - Do NOT call Notion APIs in this cell (connectivity checks belong to Cell 03).
#   - Do NOT hard-fail on optional IDs here; validate required IDs closer to usage.
#   - Notion API version is assumed >= 2025-09-03.
#

import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Load env vars (single source of truth)
load_dotenv("env.txt")

# --- Core credentials / versions ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
NOTION_TOKEN = os.getenv("NOTION_TOKEN", "")
NOTION_VERSION = os.getenv("NOTION_VERSION", "2025-09-03")

# --- Notion Data Source IDs (preferred naming aligned to your project) ---
NOTION_WEEKLY_DIGESTS_DB_ID = os.getenv("NOTION_WEEKLY_DIGESTS_DB_ID", "")
NOTION_RQ_DB_ID = os.getenv("NOTION_RQ_DB_ID", "")
NOTION_WEEKLY_RQ_UPDATE_DB_ID = os.getenv("NOTION_WEEKLY_RQ_UPDATE_DB_ID", "")
NOTION_LIT_DB_ID = os.getenv("NOTION_LIT_DB_ID", "")
NOTION_EVENTS_DB_ID = os.getenv("NOTION_EVENTS_DB_ID", "")  # if available in env.txt

# --- Time window (weekly lookback) ---
LOOKBACK_DAYS = int(os.getenv("LOOKBACK_DAYS", "7"))
NOW = datetime.now()
LOOKBACK_START = NOW - timedelta(days=LOOKBACK_DAYS)

# --- Decision filters (case-sensitive) ---
PAPER_DECISIONS_INCLUDE = {"READ", "KEEP"}

# --- Minimal config sanity hints (non-fatal here) ---
missing_core = [k for k, v in {
    "NOTION_TOKEN": NOTION_TOKEN,
    "NOTION_WEEKLY_DIGESTS_DB_ID": NOTION_WEEKLY_DIGESTS_DB_ID,
    "NOTION_RQ_DB_ID": NOTION_RQ_DB_ID,
    "NOTION_WEEKLY_RQ_UPDATE_DB_ID": NOTION_WEEKLY_RQ_UPDATE_DB_ID,
    "NOTION_LIT_DB_ID": NOTION_LIT_DB_ID,
}.items() if not v]

if missing_core:
    print("[Cell 02] WARNING: Missing core config values:", ", ".join(missing_core))
    print("          (Hard validation will occur in later cells where each value is required.)")

print("[Cell 02] Config loaded.")
print(f"  NOTION_VERSION: {NOTION_VERSION}")
print(f"  LOOKBACK_DAYS: {LOOKBACK_DAYS} (from {LOOKBACK_START.strftime('%Y-%m-%d')} to {NOW.strftime('%Y-%m-%d')})")


[Cell 02] Config loaded.
  NOTION_VERSION: 2025-09-03
  LOOKBACK_DAYS: 7 (from 2026-02-01 to 2026-02-08)


In [4]:
# ============================================================
# Cell 03 — Initialize Notion client and resolve data_source_ids (deep scan)
# ============================================================
# Overview:
#   Initialize a thin Notion HTTP client and resolve a queryable data_source_id
#   for each env-provided database_id by deep-scanning the database JSON for UUID-like
#   candidates and validating them via POST /data_sources/{id}/query.
#
# Inputs / Outputs:
#   Inputs:
#     - NOTION_TOKEN, NOTION_VERSION
#     - NOTION_*_DB_ID variables from Cell 02 (database IDs)
#   Outputs:
#     - notion_headers
#     - notion_request_json() helper (returns dict, raises on error)
#     - resolve_data_source_id_from_database() helper
#     - RESOLVED_DB: mapping name -> {"database_id": <uuid>, "data_source_id": <uuid>}
#
# Notes:
#   - This environment rejects POST /databases/{id}/query (invalid_request_url).
#   - We must query via POST /data_sources/{id}/query, but data_source_id is not easily exposed.
#   - Therefore we deep-scan the database object for UUID candidates and test them.
#

from __future__ import annotations

import re
from typing import Any, Dict, Optional, Tuple, List, Set
import requests

NOTION_API_BASE = "https://api.notion.com/v1"

if not NOTION_TOKEN:
    raise ValueError("NOTION_TOKEN is missing. Please set it in env.txt and reload Cell 02.")

notion_headers = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Content-Type": "application/json",
    "Notion-Version": NOTION_VERSION,
}

UUID_HYPHEN = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$", re.I)
HEX32 = re.compile(r"^[0-9a-f]{32}$", re.I)

def normalize_notion_id(id_like: str) -> str:
    if not id_like:
        return id_like
    s = id_like.strip().strip("{}").strip('"').strip("'")
    if HEX32.fullmatch(s):
        s = f"{s[0:8]}-{s[8:12]}-{s[12:16]}-{s[16:20]}-{s[20:32]}"
    return s.lower()

def notion_request_json(
    method: str,
    endpoint: str,
    *,
    params: Optional[Dict[str, Any]] = None,
    json_body: Optional[Dict[str, Any]] = None,
    timeout: int = 60,
) -> Dict[str, Any]:
    url = f"{NOTION_API_BASE}/{endpoint.lstrip('/')}"
    resp = requests.request(
        method.upper(),
        url,
        headers=notion_headers,
        params=params,
        json=json_body,
        timeout=timeout,
    )
    if resp.status_code >= 400:
        try:
            body = resp.json()
        except Exception:
            body = resp.text
        raise RuntimeError(f"Notion API error {resp.status_code} for {url}: {body}")
    return resp.json()

def deep_collect_ids(obj: Any) -> Set[str]:
    """
    Recursively collect UUID-like strings (hyphenated or 32-hex) from a JSON-like structure.
    """
    found: Set[str] = set()

    def _walk(x: Any):
        if isinstance(x, dict):
            for k, v in x.items():
                _walk(k)
                _walk(v)
        elif isinstance(x, list):
            for v in x:
                _walk(v)
        elif isinstance(x, str):
            s = x.strip()
            if UUID_HYPHEN.fullmatch(s) or HEX32.fullmatch(s):
                found.add(normalize_notion_id(s))

    _walk(obj)
    return found

def try_data_source_query(ds_id: str) -> bool:
    """
    Returns True if POST /data_sources/{id}/query succeeds.
    """
    try:
        _ = notion_request_json("POST", f"data_sources/{ds_id}/query", json_body={"page_size": 1})
        return True
    except Exception:
        return False

def resolve_data_source_id_from_database(db_id: str) -> str:
    """
    Given a database_id, fetch its database object and deep-scan for candidate ids,
    then validate candidates by attempting a data_sources query.
    """
    db = notion_request_json("GET", f"databases/{db_id}")
    candidates = list(deep_collect_ids(db))

    # Heuristics: try candidates closest to db_id first? (db_id itself often appears; skip it)
    candidates = [c for c in candidates if c != db_id]

    # Also try raw db_id (some environments use same id) just in case
    ordered = [db_id] + candidates

    for cand in ordered:
        if try_data_source_query(cand):
            return cand

    # If none worked, provide visibility into what was scanned
    raise ValueError(
        "Could not find a queryable data_source_id by scanning the database object.\n"
        f"Database ID: {db_id}\n"
        f"Collected {len(candidates)} UUID-like candidates, but none worked for /data_sources/{{id}}/query."
    )

print("[Cell 03] Verifying Notion connectivity...")

me = notion_request_json("GET", "users/me")
print(f"  ✓ Auth OK (users/me): {me.get('name', 'Unknown')}")

targets: List[Tuple[str, str]] = [
    ("WEEKLY_DIGESTS", NOTION_WEEKLY_DIGESTS_DB_ID),
    ("RQ_DB", NOTION_RQ_DB_ID),
    ("WEEKLY_RQ_UPDATE_DB", NOTION_WEEKLY_RQ_UPDATE_DB_ID),
    ("PAPERS_DB", NOTION_LIT_DB_ID),
    ("EVENTS_DB", NOTION_EVENTS_DB_ID),
]

missing = [n for n, v in targets if not v]
if missing:
    raise ValueError(f"Missing required DB IDs in env: {', '.join(missing)}")

RESOLVED_DB: Dict[str, Dict[str, str]] = {}

for name, raw_db_id in targets:
    db_id = normalize_notion_id(raw_db_id)

    # Confirm database is visible
    _ = notion_request_json("GET", f"databases/{db_id}")

    # Resolve a working data_source_id
    ds_id = resolve_data_source_id_from_database(db_id)

    # Verify final query
    res = notion_request_json("POST", f"data_sources/{ds_id}/query", json_body={"page_size": 1})
    n = len(res.get("results", []))

    RESOLVED_DB[name] = {"database_id": db_id, "data_source_id": ds_id}
    print(f"  ✓ {name}: db={db_id} -> data_source={ds_id} (results={n})")

print("[Cell 03] Notion client initialized and data sources resolved.")


[Cell 03] Verifying Notion connectivity...
  ✓ Auth OK (users/me): Jupyter
  ✓ WEEKLY_DIGESTS: db=2ff8e0e4-d162-8029-b1db-e2a2b9150ed4 -> data_source=2ff8e0e4-d162-8075-a3c1-000b4315606c (results=1)
  ✓ RQ_DB: db=2a98e0e4-d162-80b7-bad2-cf6635a3ef17 -> data_source=2a98e0e4-d162-8018-816d-000b5ae09450 (results=1)
  ✓ WEEKLY_RQ_UPDATE_DB: db=2ff8e0e4-d162-8039-b05c-ed10ea919ca5 -> data_source=2ff8e0e4-d162-8013-9c7b-000bf3fc12fc (results=0)
  ✓ PAPERS_DB: db=2a98e0e4-d162-80cb-b6cb-dcd1ebedee54 -> data_source=2a98e0e4-d162-801c-87a6-000bc077f4ff (results=1)
  ✓ EVENTS_DB: db=2f08e0e4-d162-80be-b40c-f607bbb3b828 -> data_source=2f08e0e4-d162-8073-8735-000bf6f8f8dd (results=1)
[Cell 03] Notion client initialized and data sources resolved.


In [5]:
# ============================================================
# Cell 04 — Introspect WEEKLY_RQ_UPDATE_DB properties
# ============================================================
# Overview:
#   Cache the property schema for WEEKLY_RQ_UPDATE_DB for runtime validation and
#   payload construction (e.g., select options, relation targets, number formats).
#
# Inputs / Outputs:
#   Inputs:
#     - notion_request_json() from Cell 03
#     - RESOLVED_DB["WEEKLY_RQ_UPDATE_DB"] containing:
#         - database_id
#         - data_source_id
#   Outputs:
#     - weekly_rq_update_db_id (str)
#     - weekly_rq_update_ds_id (str)
#     - weekly_rq_update_db_info (dict)
#     - weekly_rq_update_properties (dict)
#     - weekly_rq_update_prop_types (dict[str, str])
#     - weekly_rq_update_prop_meta (dict[str, dict])
#
# Notes:
#   - Primary source of schema is GET /v1/databases/{database_id}.
#   - If database.properties is empty, fall back to sampling a page via
#     POST /v1/data_sources/{data_source_id}/query (page_size=1).
#   - WEEKLY_RQ_UPDATE_DB may have 0 pages initially; in that case fallback cannot infer
#     properties, and we proceed with an empty schema (downstream code should handle).
#

from __future__ import annotations
from typing import Any, Dict

# --- Resolve IDs (normalized) ---
weekly_rq_update_db_id = RESOLVED_DB["WEEKLY_RQ_UPDATE_DB"]["database_id"]
weekly_rq_update_ds_id = RESOLVED_DB["WEEKLY_RQ_UPDATE_DB"]["data_source_id"]

# --- 1) Primary: schema from database metadata ---
weekly_rq_update_db_info = notion_request_json("GET", f"databases/{weekly_rq_update_db_id}")
weekly_rq_update_properties: Dict[str, Any] = weekly_rq_update_db_info.get("properties", {}) or {}

# --- 2) Fallback: infer property keys from a sampled page if properties are empty ---
if not weekly_rq_update_properties:
    print("[Cell 04] WARNING: database.properties is empty; trying to infer schema from a sample page via data_sources.")
    sample = notion_request_json(
        "POST",
        f"data_sources/{weekly_rq_update_ds_id}/query",
        json_body={"page_size": 1},
    )
    results = sample.get("results", []) or []
    if results:
        weekly_rq_update_properties = results[0].get("properties", {}) or {}
    else:
        # Common when the DB has no pages yet
        weekly_rq_update_properties = {}
        print("[Cell 04] NOTE: No pages found in WEEKLY_RQ_UPDATE_DB; schema inference from sample page is not possible.")

# --- 3) Catalog property types + lightweight metadata for runtime checks ---
weekly_rq_update_prop_types: Dict[str, str] = {}
weekly_rq_update_prop_meta: Dict[str, Dict[str, Any]] = {}

def _safe_uuid(s: Any) -> str:
    if not s or not isinstance(s, str):
        return "N/A"
    return s if len(s) <= 36 else s[:36]

for prop_name, prop_config in weekly_rq_update_properties.items():
    prop_type = prop_config.get("type") if isinstance(prop_config, dict) else None
    weekly_rq_update_prop_types[prop_name] = prop_type or "unknown"

    meta: Dict[str, Any] = {}
    if prop_type == "select":
        meta["options"] = [opt.get("name") for opt in (prop_config.get("select", {}) or {}).get("options", [])]
    elif prop_type == "multi_select":
        meta["options"] = [opt.get("name") for opt in (prop_config.get("multi_select", {}) or {}).get("options", [])]
    elif prop_type == "relation":
        rel = prop_config.get("relation", {}) or {}
        meta["related_database_id"] = rel.get("database_id")
        meta["synced_property_name"] = rel.get("synced_property_name")
    elif prop_type == "number":
        meta["format"] = (prop_config.get("number", {}) or {}).get("format")

    weekly_rq_update_prop_meta[prop_name] = meta

# --- 4) Print summary (compact but useful) ---
print("WEEKLY_RQ_UPDATE_DB Property Schema")
print(f"  database_id: {weekly_rq_update_db_id}")
print(f"  data_source_id: {weekly_rq_update_ds_id}")
print(f"  total properties discovered: {len(weekly_rq_update_prop_types)}\n")

for prop_name in sorted(weekly_rq_update_prop_types.keys()):
    t = weekly_rq_update_prop_types[prop_name]
    print(f"  - {prop_name}: {t}")

    meta = weekly_rq_update_prop_meta.get(prop_name, {})
    if t in {"select", "multi_select"} and meta.get("options"):
        print(f"      options: {meta['options']}")
    if t == "relation":
        print(f"      related_database_id: {_safe_uuid(meta.get('related_database_id'))}")
        if meta.get("synced_property_name"):
            print(f"      synced_property_name: {meta['synced_property_name']}")
    if t == "number" and meta.get("format"):
        print(f"      format: {meta['format']}")

print("\n[Cell 04] Cached WEEKLY_RQ_UPDATE_DB schema for runtime use.")


[Cell 04] WARNING: database.properties is empty; trying to infer schema from a sample page via data_sources.
[Cell 04] NOTE: No pages found in WEEKLY_RQ_UPDATE_DB; schema inference from sample page is not possible.
WEEKLY_RQ_UPDATE_DB Property Schema
  database_id: 2ff8e0e4-d162-8039-b05c-ed10ea919ca5
  data_source_id: 2ff8e0e4-d162-8013-9c7b-000bf3fc12fc
  total properties discovered: 0


[Cell 04] Cached WEEKLY_RQ_UPDATE_DB schema for runtime use.


In [6]:
# ============================================================
# Cell 05 — Introspect RQ_DB properties
# ============================================================
# Overview:
#   Cache the property schema (or inferred property types) for RQ_DB.
#   Prefer database metadata schema; if unavailable, infer types from a sample page.
#
# Inputs / Outputs:
#   Inputs:
#     - notion_request_json() from Cell 03
#     - RESOLVED_DB["RQ_DB"] containing database_id + data_source_id
#   Outputs:
#     - rq_db_id, rq_ds_id
#     - rq_db_info (dict)
#     - rq_properties (dict)               # schema properties if available; else {}
#     - rq_prop_types (dict[str, str])     # always populated when possible
#     - rq_prop_meta (dict[str, dict])     # only reliable when schema is available
#     - rq_schema_source (str)             # "database_metadata" or "sample_page_inference"
#
# Notes:
#   - If schema is inferred from a sample page, only "type" is trusted.
#     Options and relation target database IDs are not reliably inferable.
#

from __future__ import annotations
from typing import Any, Dict

rq_db_id = RESOLVED_DB["RQ_DB"]["database_id"]
rq_ds_id = RESOLVED_DB["RQ_DB"]["data_source_id"]

rq_db_info = notion_request_json("GET", f"databases/{rq_db_id}")
rq_properties: Dict[str, Any] = rq_db_info.get("properties", {}) or {}

rq_prop_types: Dict[str, str] = {}
rq_prop_meta: Dict[str, Dict[str, Any]] = {}
rq_schema_source: str = "database_metadata"

def _infer_types_from_page_properties(page_props: Dict[str, Any]) -> Dict[str, str]:
    inferred: Dict[str, str] = {}
    for name, obj in (page_props or {}).items():
        if isinstance(obj, dict) and "type" in obj:
            inferred[name] = obj.get("type") or "unknown"
        else:
            inferred[name] = "unknown"
    return inferred

# --- 1) If schema exists, parse it (rich meta) ---
if rq_properties:
    for prop_name, prop_config in rq_properties.items():
        prop_type = prop_config.get("type") if isinstance(prop_config, dict) else None
        rq_prop_types[prop_name] = prop_type or "unknown"

        meta: Dict[str, Any] = {}
        if prop_type == "select":
            meta["options"] = [opt.get("name") for opt in (prop_config.get("select", {}) or {}).get("options", [])]
        elif prop_type == "multi_select":
            meta["options"] = [opt.get("name") for opt in (prop_config.get("multi_select", {}) or {}).get("options", [])]
        elif prop_type == "relation":
            rel = prop_config.get("relation", {}) or {}
            # In schema, relation is a dict. (If not, keep meta empty)
            if isinstance(rel, dict):
                meta["related_database_id"] = rel.get("database_id")
                meta["synced_property_name"] = rel.get("synced_property_name")
        elif prop_type == "number":
            meta["format"] = (prop_config.get("number", {}) or {}).get("format")

        rq_prop_meta[prop_name] = meta

else:
    # --- 2) Fallback: infer types from a sample page (type only) ---
    rq_schema_source = "sample_page_inference"
    print("[Cell 05] WARNING: database.properties is empty; inferring property types from a sample page via data_sources.")

    sample = notion_request_json("POST", f"data_sources/{rq_ds_id}/query", json_body={"page_size": 1})
    results = sample.get("results", []) or []
    if results:
        page_props = results[0].get("properties", {}) or {}
        rq_prop_types = _infer_types_from_page_properties(page_props)
        rq_prop_meta = {k: {} for k in rq_prop_types.keys()}  # meta not reliable in inference mode
    else:
        print("[Cell 05] NOTE: No pages found in RQ_DB; cannot infer schema/types.")
        rq_prop_types = {}
        rq_prop_meta = {}

# --- 3) Print summary ---
print("RQ_DB Property Schema (cached)")
print(f"  database_id: {rq_db_id}")
print(f"  data_source_id: {rq_ds_id}")
print(f"  schema_source: {rq_schema_source}")
print(f"  total properties discovered: {len(rq_prop_types)}\n")

for prop_name in sorted(rq_prop_types.keys()):
    t = rq_prop_types[prop_name]
    print(f"  - {prop_name}: {t}")

    # Only print meta details when schema source is metadata
    if rq_schema_source == "database_metadata":
        meta = rq_prop_meta.get(prop_name, {})
        if t in {"select", "multi_select"} and meta.get("options"):
            print(f"      options: {meta['options']}")
        if t == "relation" and meta.get("related_database_id"):
            print(f"      related_database_id: {meta.get('related_database_id')}")
        if t == "number" and meta.get("format"):
            print(f"      format: {meta.get('format')}")

print("\n[Cell 05] Cached RQ_DB types for runtime use.")


[Cell 05] WARNING: database.properties is empty; inferring property types from a sample page via data_sources.
RQ_DB Property Schema (cached)
  database_id: 2a98e0e4-d162-80b7-bad2-cf6635a3ef17
  data_source_id: 2a98e0e4-d162-8018-816d-000b5ae09450
  schema_source: sample_page_inference
  total properties discovered: 8

  - Gap Identified: rich_text
  - Linked Paper: relation
  - Name: title
  - Priority: select
  - Proposed Approach: rich_text
  - Rationale / Background: rich_text
  - Status: status
  - Tags: multi_select

[Cell 05] Cached RQ_DB types for runtime use.


In [7]:
# ============================================================
# Cell 06 — Introspect PAPERS_DB properties
# ============================================================
# Overview:
#   Cache the property schema (or inferred property types) for PAPERS_DB.
#   This is used to build robust filters (Decision=READ/KEEP, recent ingestion window)
#   and to extract evidence fields for RQ updates.
#
# Inputs / Outputs:
#   Inputs:
#     - notion_request_json() from Cell 03
#     - RESOLVED_DB["PAPERS_DB"] containing database_id + data_source_id
#     - (Optional) PAPERS_SCHEMA constant (reference)
#   Outputs:
#     - papers_db_id, papers_ds_id
#     - papers_db_info (dict)
#     - papers_properties (dict)                 # schema properties if available; else {}
#     - papers_prop_types (dict[str, str])       # always populated when possible
#     - papers_prop_meta (dict[str, dict])       # only reliable when schema is available
#     - papers_schema_source (str)               # "database_metadata" or "sample_page_inference"
#
# Notes:
#   - Primary source is GET /v1/databases/{database_id}.
#   - If database.properties is empty, infer types from a sample page via
#     POST /v1/data_sources/{data_source_id}/query (page_size=1).
#   - In inference mode, only "type" is trusted. Options / relation targets are not reliable.
#

from __future__ import annotations
from typing import Any, Dict, List

papers_db_id = RESOLVED_DB["PAPERS_DB"]["database_id"]
papers_ds_id = RESOLVED_DB["PAPERS_DB"]["data_source_id"]

papers_db_info = notion_request_json("GET", f"databases/{papers_db_id}")
papers_properties: Dict[str, Any] = papers_db_info.get("properties", {}) or {}

papers_prop_types: Dict[str, str] = {}
papers_prop_meta: Dict[str, Dict[str, Any]] = {}
papers_schema_source: str = "database_metadata"

def _infer_types_from_page_properties(page_props: Dict[str, Any]) -> Dict[str, str]:
    inferred: Dict[str, str] = {}
    for name, obj in (page_props or {}).items():
        if isinstance(obj, dict) and "type" in obj:
            inferred[name] = obj.get("type") or "unknown"
        else:
            inferred[name] = "unknown"
    return inferred

# --- 1) If schema exists, parse it (rich meta) ---
if papers_properties:
    for prop_name, prop_config in papers_properties.items():
        prop_type = prop_config.get("type") if isinstance(prop_config, dict) else None
        papers_prop_types[prop_name] = prop_type or "unknown"

        meta: Dict[str, Any] = {}
        if prop_type == "select":
            meta["options"] = [opt.get("name") for opt in (prop_config.get("select", {}) or {}).get("options", [])]
        elif prop_type == "multi_select":
            meta["options"] = [opt.get("name") for opt in (prop_config.get("multi_select", {}) or {}).get("options", [])]
        elif prop_type == "relation":
            rel = prop_config.get("relation", {}) or {}
            if isinstance(rel, dict):
                meta["related_database_id"] = rel.get("database_id")
                meta["synced_property_name"] = rel.get("synced_property_name")
        elif prop_type == "number":
            meta["format"] = (prop_config.get("number", {}) or {}).get("format")

        papers_prop_meta[prop_name] = meta

else:
    # --- 2) Fallback: infer types from a sample page (type only) ---
    papers_schema_source = "sample_page_inference"
    print("[Cell 06] WARNING: database.properties is empty; inferring property types from a sample page via data_sources.")

    sample = notion_request_json("POST", f"data_sources/{papers_ds_id}/query", json_body={"page_size": 1})
    results = sample.get("results", []) or []
    if results:
        page_props = results[0].get("properties", {}) or {}
        papers_prop_types = _infer_types_from_page_properties(page_props)
        papers_prop_meta = {k: {} for k in papers_prop_types.keys()}  # meta not reliable in inference mode
    else:
        print("[Cell 06] NOTE: No pages found in PAPERS_DB; cannot infer schema/types.")
        papers_prop_types = {}
        papers_prop_meta = {}

# --- 3) Print summary (compact) ---
print("PAPERS_DB Property Schema (cached)")
print(f"  database_id: {papers_db_id}")
print(f"  data_source_id: {papers_ds_id}")
print(f"  schema_source: {papers_schema_source}")
print(f"  total properties discovered: {len(papers_prop_types)}\n")

for prop_name in sorted(papers_prop_types.keys()):
    t = papers_prop_types[prop_name]
    print(f"  - {prop_name}: {t}")

    if papers_schema_source == "database_metadata":
        meta = papers_prop_meta.get(prop_name, {})
        if t in {"select", "multi_select"} and meta.get("options"):
            print(f"      options: {meta['options']}")
        if t == "relation" and meta.get("related_database_id"):
            print(f"      related_database_id: {meta.get('related_database_id')}")
        if t == "number" and meta.get("format"):
            print(f"      format: {meta.get('format')}")

# --- 4) Light sanity check for downstream-required fields (warn only) ---
expected_for_downstream: List[str] = ["Name", "Ingested At", "Decision", "Decision Reason", "RQ Relevance", "Weekly Priority"]
missing_expected = [p for p in expected_for_downstream if p not in papers_prop_types]

if missing_expected:
    print("\n[Cell 06] WARNING: Some expected paper properties were not detected:")
    for p in missing_expected:
        print(f"  - {p}")
    print("  (This may be OK if the sample page did not include them; verify later if filters fail.)")

print("\n[Cell 06] Cached PAPERS_DB types for runtime use.")


[Cell 06] WARNING: database.properties is empty; inferring property types from a sample page via data_sources.
PAPERS_DB Property Schema (cached)
  database_id: 2a98e0e4-d162-80cb-b6cb-dcd1ebedee54
  data_source_id: 2a98e0e4-d162-801c-87a6-000bc077f4ff
  schema_source: sample_page_inference
  total properties discovered: 25

  - Authors & Year: rich_text
  - Core Idea: rich_text
  - Created time: created_time
  - Datasets: rich_text
  - Decision: select
  - Decision Reason: rich_text
  - Dedup Key: rich_text
  - Findings: rich_text
  - Importance: number
  - Ingested At: date
  - Methods: rich_text
  - Name: title
  - Notes: rich_text
  - PDF Link: url
  - PDF Status: select
  - Papers: relation
  - RQ Relevance: number
  - Run ID: rich_text
  - Slide 1 URL: url
  - Source: rich_text
  - Source UID: rich_text
  - Status: select
  - Tags: multi_select
  - Type: rich_text
  - Weekly Priority: number

[Cell 06] Cached PAPERS_DB types for runtime use.


In [8]:
# ============================================================
# Cell 07 — Load most recent weekly digest record
# ============================================================
# Overview:
#   Load the most recent Weekly Digest page to serve as the immutable weekly context.
#   This page will later be updated ONLY by writing the "RQ Updates" relation.
#
# Inputs / Outputs:
#   Inputs:
#     - notion_request_json() from Cell 03
#     - RESOLVED_DB["WEEKLY_DIGESTS"] (database_id + data_source_id)
#     - WEEKLY_DIGESTS_DB schema expectations (Name, Week Start, Week End, Run ID, Generated At, Status, RQ Updates)
#   Outputs:
#     - weekly_digest (dict): the selected digest page object
#     - weekly_digest_id (str)
#     - digest_title (str)
#     - digest_week_start (str|None)
#     - digest_week_end (str|None)
#     - digest_run_id (str|None)
#     - digest_generated_at (str|None)
#     - digest_status (str|None)
#     - existing_rq_updates (list[str])  # page IDs currently related
#
# Notes:
#   - Query uses POST /v1/data_sources/{id}/query (database query endpoint is invalid in this environment).
#   - "Most recent" is determined by sort priority:
#       1) Generated At (descending)
#       2) Week End (descending)
#       3) created_time (descending)
#   - If a property name differs in your DB, update the property keys below.
#

from __future__ import annotations
from typing import Any, Dict, List, Optional

weekly_digests_db_id = RESOLVED_DB["WEEKLY_DIGESTS"]["database_id"]
weekly_digests_ds_id = RESOLVED_DB["WEEKLY_DIGESTS"]["data_source_id"]

def get_prop(page: Dict[str, Any], prop_name: str) -> Dict[str, Any]:
    return (page.get("properties", {}) or {}).get(prop_name, {}) or {}

def read_title(page: Dict[str, Any], prop_name: str = "Name") -> str:
    p = get_prop(page, prop_name)
    if p.get("type") != "title":
        return ""
    parts = p.get("title", []) or []
    return "".join([t.get("plain_text", "") for t in parts]).strip()

def read_rich_text(page: Dict[str, Any], prop_name: str) -> str:
    p = get_prop(page, prop_name)
    if p.get("type") != "rich_text":
        return ""
    parts = p.get("rich_text", []) or []
    return "".join([t.get("plain_text", "") for t in parts]).strip()

def read_date_start(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    p = get_prop(page, prop_name)
    if p.get("type") != "date":
        return None
    d = p.get("date")
    return (d or {}).get("start")

def read_select_name(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    p = get_prop(page, prop_name)
    t = p.get("type")
    if t == "select":
        sel = p.get("select")
        return (sel or {}).get("name")
    if t == "status":  # Notion "status" type
        st = p.get("status")
        return (st or {}).get("name")
    return None

def read_relation_ids(page: Dict[str, Any], prop_name: str) -> List[str]:
    p = get_prop(page, prop_name)
    if p.get("type") != "relation":
        return []
    rel = p.get("relation", []) or []
    return [r.get("id") for r in rel if r.get("id")]

# ---- Query the most recent digest ----
query_payload = {
    "sorts": [
        {"property": "Generated At", "direction": "descending"},
        {"property": "Week End", "direction": "descending"},
        {"timestamp": "created_time", "direction": "descending"},
    ],
    "page_size": 1,
}

response = notion_request_json(
    "POST",
    f"data_sources/{weekly_digests_ds_id}/query",
    json_body=query_payload,
)

results = response.get("results", []) or []
if not results:
    raise ValueError("No weekly digest records found in WEEKLY_DIGESTS_DB (data source query returned 0 results).")

weekly_digest = results[0]
weekly_digest_id = weekly_digest.get("id")

# ---- Extract context fields (adjust names if your DB differs) ----
digest_title = read_title(weekly_digest, "Name")
digest_week_start = read_date_start(weekly_digest, "Week Start")
digest_week_end = read_date_start(weekly_digest, "Week End")
digest_run_id = read_rich_text(weekly_digest, "Run ID")
digest_generated_at = read_date_start(weekly_digest, "Generated At")
digest_status = read_select_name(weekly_digest, "Status")
existing_rq_updates = read_relation_ids(weekly_digest, "RQ Updates")

print("Most recent weekly digest loaded:")
print(f"  ID: {weekly_digest_id}")
print(f"  Title: {digest_title}")
print(f"  Week Start: {digest_week_start}")
print(f"  Week End: {digest_week_end}")
print(f"  Run ID: {digest_run_id}")
print(f"  Generated At: {digest_generated_at}")
print(f"  Status: {digest_status}")
print(f"  Existing RQ Updates: {len(existing_rq_updates)} relations")

print("\n[Cell 07] Weekly digest cached for downstream RQ update linking (only 'RQ Updates' will be written later).")


Most recent weekly digest loaded:
  ID: 2ff8e0e4-d162-8165-9a05-e4061df44fb5
  Title: Weekly Events Digest: 2026-02-02–2026-02-08
  Week Start: 2026-02-02
  Week End: 2026-02-08
  Run ID: 20260206_211807
  Generated At: 2026-02-06T21:18:00.000+09:00
  Status: generated
  Existing RQ Updates: 0 relations

[Cell 07] Weekly digest cached for downstream RQ update linking (only 'RQ Updates' will be written later).


In [9]:
# ============================================================
# Cell 08 — Load high-priority research questions from RQ_DB
# ============================================================
# Overview:
#   Load Research Questions (RQs) from RQ_DB and select the subset to update this week.
#   This notebook prioritizes focus: we primarily target RQs whose Priority is "High".
#
# Inputs / Outputs:
#   Inputs:
#     - notion_request_json() from Cell 03
#     - RESOLVED_DB["RQ_DB"] (database_id + data_source_id)
#     - rq_prop_types from Cell 05 (optional; used for awareness only)
#   Outputs:
#     - all_rqs (list[dict])        : all RQ pages fetched (before filtering)
#     - target_rqs (list[dict])     : pages selected for this week (Priority=High)
#     - rq_records (list[dict])     : structured records for downstream processing
#
# Notes:
#   - Query uses POST /v1/data_sources/{id}/query (database query endpoint is invalid in this environment).
#   - We intentionally filter in Python (rather than Notion-side filters) to avoid
#     type mismatches (e.g., Priority/Status might be 'select' or 'status' across workspaces).
#   - Priority values are case-sensitive; expected target is "High".
#   - Status is captured as context, but not used for filtering.
#

from __future__ import annotations
from typing import Any, Dict, List, Optional

rq_db_id = RESOLVED_DB["RQ_DB"]["database_id"]
rq_ds_id = RESOLVED_DB["RQ_DB"]["data_source_id"]

# --- Helpers to read Notion page properties safely ---
def _get_prop(page: Dict[str, Any], prop_name: str) -> Dict[str, Any]:
    return (page.get("properties", {}) or {}).get(prop_name, {}) or {}

def _read_title(page: Dict[str, Any], prop_name: str = "Name") -> str:
    p = _get_prop(page, prop_name)
    if p.get("type") != "title":
        return ""
    parts = p.get("title", []) or []
    return "".join([t.get("plain_text", "") for t in parts]).strip()

def _read_rich_text(page: Dict[str, Any], prop_name: str) -> str:
    p = _get_prop(page, prop_name)
    if p.get("type") != "rich_text":
        return ""
    parts = p.get("rich_text", []) or []
    return "".join([t.get("plain_text", "") for t in parts]).strip()

def _read_select_name(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    p = _get_prop(page, prop_name)
    t = p.get("type")
    if t == "select":
        sel = p.get("select") or {}
        return sel.get("name")
    if t == "status":  # Notion "status" type
        st = p.get("status") or {}
        return st.get("name")
    return None

def _read_multi_select(page: Dict[str, Any], prop_name: str) -> List[str]:
    p = _get_prop(page, prop_name)
    if p.get("type") != "multi_select":
        return []
    items = p.get("multi_select", []) or []
    return [it.get("name") for it in items if it.get("name")]

# --- Define which RQs to update this week ---
TARGET_PRIORITIES = {"High"}  # focus set for Weekly (case-sensitive)

# --- Query all RQs (no Notion-side filter to avoid type mismatch) ---
query_payload: Dict[str, Any] = {
    "sorts": [{"timestamp": "created_time", "direction": "descending"}],
    "page_size": 100,
}

all_rqs: List[Dict[str, Any]] = []
has_more = True
start_cursor = None

while has_more:
    payload = dict(query_payload)
    if start_cursor:
        payload["start_cursor"] = start_cursor

    resp = notion_request_json("POST", f"data_sources/{rq_ds_id}/query", json_body=payload)
    batch = resp.get("results", []) or []
    all_rqs.extend(batch)

    has_more = bool(resp.get("has_more"))
    start_cursor = resp.get("next_cursor")

print(f"[Cell 08] RQs fetched (total, before filtering): {len(all_rqs)}")
print(f"[Cell 08] Target priorities: {sorted(TARGET_PRIORITIES)}")

# --- Filter by Priority in Python ---
target_rqs: List[Dict[str, Any]] = []
for page in all_rqs:
    priority = _read_select_name(page, "Priority")
    if priority in TARGET_PRIORITIES:
        target_rqs.append(page)

print(f"[Cell 08] RQs selected for this week: {len(target_rqs)}")

# --- Build structured records for downstream processing ---
rq_records: List[Dict[str, Any]] = []

for rq_page in target_rqs:
    rq_id = rq_page.get("id")
    rq_title = _read_title(rq_page, "Name")
    rq_status = _read_select_name(rq_page, "Status")      # captured for context only
    rq_priority = _read_select_name(rq_page, "Priority")  # expected "High"
    rq_tags = _read_multi_select(rq_page, "Tags")

    # Choose a "description-ish" field if present; fallback chain
    rq_description = ""
    for candidate in ["Description", "Rationale / Background", "Proposed Approach", "Gap Identified"]:
        txt = _read_rich_text(rq_page, candidate)
        if txt:
            rq_description = txt
            break

    rq_records.append(
        {
            "id": rq_id,
            "title": rq_title,
            "status": rq_status,
            "priority": rq_priority,
            "tags": rq_tags,
            "description": rq_description,
            "page": rq_page,  # keep full page for later linking
        }
    )

# --- Print compact preview (up to 30) ---
for r in rq_records[:30]:
    tags_str = ", ".join(r["tags"]) if r["tags"] else "None"
    print(f"  - {r['title']}  | Priority: {r['priority']} | Status: {r['status']} | Tags: {tags_str}")

if len(rq_records) > 30:
    print(f"  ... ({len(rq_records) - 30} more)")

print(f"\n[Cell 08] Total RQs cached for weekly updates (Priority=High): {len(rq_records)}")


[Cell 08] RQs fetched (total, before filtering): 60
[Cell 08] Target priorities: ['High']
[Cell 08] RQs selected for this week: 8
  - [v1.1] スタートアップ投資バブルを長期的なイノベーションの軌道に変換するための神話や物語構造と、CVC戦略の役割は何か？  | Priority: High | Status: New | Tags: None
  - [v1.1] 外国系VCとコーポレートVCは、新興市場におけるスタートアップの成長とイノベーション成果にどのように異なる影響を与えるか？  | Priority: High | Status: New | Tags: Emerging Markets
  - LP投資者として行動する政府系ファンド（SWF）は、国内スタートアップ・エコシステムの形成にどのような影響を与えるか？  | Priority: High | Status: New | Tags: None
  - スタートアップ投資バブルを長期的なイノベーションの軌道に変換するには、どのような神話や物語構造が必要か？  | Priority: High | Status: New | Tags: None
  - 官民連携型のイノベーションハブがエコシステムの中核機能を果たすためには、どのような制度的要因（法制度、文化、統治能力）が必要か？  | Priority: High | Status: New | Tags: Institutions
  - 外国系VCとコーポレートVCは、新興エコシステムにおけるスタートアップの成果にどのような違いをもたらすか？  | Priority: High | Status: New | Tags: Emerging Markets
  - 政府の間接的VC（LP出資）モデルは直接VCより効果的か？  | Priority: High | Status: Under Review | Tags: Indirect VC, Institutional Quality
  - どの公的介入が民間ベンチャーキャピタルの呼び込みに最も効果的か？  | Priority: High | Stat

In [10]:
# ============================================================
# Cell 09 — Load recent papers with Decision READ or KEEP
# ============================================================
# Overview:
#   Load evidence papers from PAPERS_DB within the last LOOKBACK_DAYS whose Decision is
#   exactly "READ" or "KEEP" (uppercase). These papers will be used as evidence when
#   drafting weekly RQ updates and linking relations.
#
# Inputs / Outputs:
#   Inputs:
#     - notion_request_json() from Cell 03
#     - RESOLVED_DB["PAPERS_DB"] (database_id + data_source_id)
#     - LOOKBACK_DAYS (int) from Cell 02 (default 7)
#     - Optional: lookback_date (YYYY-MM-DD string) from Cell 02
#   Outputs:
#     - all_evidence_papers (list[dict])     : raw Notion page objects
#     - paper_records (list[dict])           : structured paper records for downstream use
#     - lookback_date_str (str)              : normalized YYYY-MM-DD used in the query
#
# Notes:
#   - Query uses POST /v1/data_sources/{id}/query (database query endpoint is invalid in this environment).
#   - We run two queries (READ, KEEP) to keep filter logic explicit.
#   - In your schema, ingestion date is "Ingested At" (NOT "Date Ingested").
#   - This cell defensively normalizes lookback_date to a YYYY-MM-DD string (prevents validation_error).
#

from __future__ import annotations
from typing import Any, Dict, List, Optional
from datetime import datetime, timedelta

papers_db_id = RESOLVED_DB["PAPERS_DB"]["database_id"]
papers_ds_id = RESOLVED_DB["PAPERS_DB"]["data_source_id"]

# ---- Configure property names (edit here if your Notion names differ) ----
DECISION_PROP = "Decision"
INGESTED_PROP = "Ingested At"        # per your schema
TITLE_PROP = "Name"
AUTHORS_PROP = "Authors & Year"      # per your schema
TAGS_PROP = "Tags"
SUMMARY_PROP_CANDIDATES = ["Findings", "Core Idea", "Notes"]

# ---- Normalize lookback_date to YYYY-MM-DD ----
def _to_yyyy_mm_dd(x: Any) -> Optional[str]:
    """
    Accept:
      - 'YYYY-MM-DD' string
      - datetime
    Return:
      - 'YYYY-MM-DD' string or None
    """
    if x is None:
        return None
    if isinstance(x, str):
        s = x.strip()
        # accept YYYY-MM-DD or ISO date-time; keep only the date part
        if len(s) >= 10 and s[4] == "-" and s[7] == "-":
            return s[:10]
        return None
    if isinstance(x, datetime):
        return x.strftime("%Y-%m-%d")
    return None

# Prefer an existing string if provided; otherwise compute from LOOKBACK_DAYS
lookback_date_str = _to_yyyy_mm_dd(globals().get("lookback_date"))
if not lookback_date_str:
    # LOOKBACK_DAYS should be int; default to 7 if missing/malformed
    _days = globals().get("LOOKBACK_DAYS")
    if not isinstance(_days, int):
        _days = 7
    lookback_date_str = (datetime.now() - timedelta(days=_days)).strftime("%Y-%m-%d")

print(f"[Cell 09] Using lookback_date (YYYY-MM-DD): {lookback_date_str}")

# ---- Helpers to read Notion page properties safely ----
def _get_prop(page: Dict[str, Any], prop_name: str) -> Dict[str, Any]:
    return (page.get("properties", {}) or {}).get(prop_name, {}) or {}

def _read_title(page: Dict[str, Any], prop_name: str = TITLE_PROP) -> str:
    p = _get_prop(page, prop_name)
    if p.get("type") != "title":
        return ""
    parts = p.get("title", []) or []
    return "".join([t.get("plain_text", "") for t in parts]).strip()

def _read_rich_text(page: Dict[str, Any], prop_name: str) -> str:
    p = _get_prop(page, prop_name)
    if p.get("type") != "rich_text":
        return ""
    parts = p.get("rich_text", []) or []
    return "".join([t.get("plain_text", "") for t in parts]).strip()

def _read_select_name(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    p = _get_prop(page, prop_name)
    t = p.get("type")
    if t == "select":
        sel = p.get("select") or {}
        return sel.get("name")
    if t == "status":
        st = p.get("status") or {}
        return st.get("name")
    return None

def _read_multi_select(page: Dict[str, Any], prop_name: str) -> List[str]:
    p = _get_prop(page, prop_name)
    if p.get("type") != "multi_select":
        return []
    items = p.get("multi_select", []) or []
    return [it.get("name") for it in items if it.get("name")]

def _read_date_start(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    p = _get_prop(page, prop_name)
    if p.get("type") != "date":
        return None
    d = p.get("date") or {}
    return d.get("start")

# ---- Query builder ----
def _query_papers_by_decision(decision_value: str) -> List[Dict[str, Any]]:
    """
    Query papers where:
      - Decision == decision_value (READ or KEEP)
      - Ingested At >= lookback_date_str (YYYY-MM-DD)
    """
    query_payload: Dict[str, Any] = {
        "filter": {
            "and": [
                {"property": DECISION_PROP, "select": {"equals": decision_value}},
                {"property": INGESTED_PROP, "date": {"on_or_after": lookback_date_str}},
            ]
        },
        "sorts": [{"property": INGESTED_PROP, "direction": "descending"}],
        "page_size": 100,
    }

    out: List[Dict[str, Any]] = []
    has_more = True
    start_cursor = None

    while has_more:
        payload = dict(query_payload)
        if start_cursor:
            payload["start_cursor"] = start_cursor

        resp = notion_request_json("POST", f"data_sources/{papers_ds_id}/query", json_body=payload)
        batch = resp.get("results", []) or []
        out.extend(batch)

        has_more = bool(resp.get("has_more"))
        start_cursor = resp.get("next_cursor")

    return out

# ---- Run queries ----
read_papers = _query_papers_by_decision("READ")
print(f"[Cell 09] Papers loaded (Decision=READ): {len(read_papers)}")

keep_papers = _query_papers_by_decision("KEEP")
print(f"[Cell 09] Papers loaded (Decision=KEEP): {len(keep_papers)}")

all_evidence_papers = read_papers + keep_papers
print(f"[Cell 09] Total evidence papers (READ+KEEP): {len(all_evidence_papers)}")

# ---- Build structured paper records (EXTENDED for RQ reasoning) ----
paper_records: List[Dict[str, Any]] = []

for paper_page in all_evidence_papers:
    paper_id = paper_page.get("id")
    paper_title = _read_title(paper_page, TITLE_PROP)
    paper_decision = _read_select_name(paper_page, DECISION_PROP)
    paper_ingested = _read_date_start(paper_page, INGESTED_PROP)
    paper_authors = _read_rich_text(paper_page, AUTHORS_PROP)
    paper_tags = _read_multi_select(paper_page, TAGS_PROP)

    # --- Rich content from LIT_DB (used for RQ Rationale / Approach / Gap) ---
    paper_findings = _read_rich_text(paper_page, "Findings")
    paper_core_idea = _read_rich_text(paper_page, "Core Idea")
    paper_notes = _read_rich_text(paper_page, "Notes")
    paper_methods = _read_rich_text(paper_page, "Methods")
    paper_datasets = _read_rich_text(paper_page, "Datasets")

    # --- Backward-compatible summary (best-effort, keep for display/debug) ---
    paper_summary = ""
    for c in SUMMARY_PROP_CANDIDATES:
        paper_summary = _read_rich_text(paper_page, c)
        if paper_summary:
            break

    paper_records.append(
        {
            "id": paper_id,
            "title": paper_title,
            "decision": paper_decision,
            "ingested": paper_ingested,
            "authors_year": paper_authors,
            "tags": paper_tags,

            # --- NEW: semantic fields for LLM reasoning ---
            "findings": paper_findings,
            "core_idea": paper_core_idea,
            "notes": paper_notes,
            "methods": paper_methods,
            "datasets": paper_datasets,

            # --- legacy / convenience ---
            "summary": paper_summary,

            # keep full page if needed later
            "page": paper_page,
        }
    )

# ---- Print compact preview ----
for p in paper_records[:20]:
    tags_str = ", ".join(p["tags"]) if p["tags"] else "None"
    print(f"  - {p['title'][:90]}")
    print(f"      Decision: {p['decision']} | Ingested At: {p['ingested']} | Tags: {tags_str}")
    if p["core_idea"]:
        print(f"      Core Idea: {p['core_idea'][:80]}...")
    if p["methods"]:
        print(f"      Methods: {p['methods'][:80]}...")
    if p["datasets"]:
        print(f"      Datasets: {p['datasets'][:80]}...")

if len(paper_records) > 20:
    print(f"  ... ({len(paper_records) - 20} more)")

print(f"\n[Cell 09] Total evidence papers cached: {len(paper_records)}")

[Cell 09] Using lookback_date (YYYY-MM-DD): 2026-02-01
[Cell 09] Papers loaded (Decision=READ): 5
[Cell 09] Papers loaded (Decision=KEEP): 2
[Cell 09] Total evidence papers (READ+KEEP): 7
  - Solving the Problem of Abundance: Venture Capital and the Making of Asset-Driven Inequalit
      Decision: READ | Ingested At: 2026-02-04 | Tags: Venture Capital, Financialization, Asset Inequality, Limited Partnership, Institutional Investors
      Core Idea: 本論文は、1950年代から1970年代にかけて米国のベンチャーキャピタル（VC）が豊富な資本を迅速に資産に変換するための金融インフラを構築し、これが資産主導の不...
      Methods: ラジカル・ヒストリシズムを用いた歴史的・概念的分析。二次文献と口述歴史インタビューの組み合わせによるナラティブ構築。...
      Datasets: 本研究は、米国におけるベンチャーキャピタルの起源に関する二次文献と、Bancroft Library Oral History Archiveの初期ベンチャーキ...
  - From D.C. to VC: Leveraging Government Expertise in Venture Capital
      Decision: READ | Ingested At: 2026-02-04 | Tags: Venture Capital, Government Expertise, Startup Financing, Public Funding, Revolving Door, Innovation Policy
      Core Idea: 元政府高官がベンチャーキャピタルに参画することで、スタートアップが政

In [11]:
# ============================================================
# Cell 10 — Load weekly events for evidence linking
# ============================================================
# Overview:
#   Load Events from EVENTS_DB within the weekly digest time window (Week Start..Week End)
#   to use as evidence when drafting and linking weekly RQ updates.
#
# Inputs / Outputs:
#   Inputs:
#     - notion_request_json() from Cell 03
#     - RESOLVED_DB["EVENTS_DB"] (database_id + data_source_id)
#     - digest_week_start, digest_week_end from Cell 07 (YYYY-MM-DD or ISO)
#     - lookback_date_str from Cell 09 (YYYY-MM-DD) as fallback if digest dates missing
#   Outputs:
#     - all_events (list[dict])     : raw Notion page objects
#     - event_records (list[dict])  : structured event records for downstream use
#
# Notes:
#   - Query uses POST /v1/data_sources/{id}/query (database query endpoint is invalid in this environment).
#   - Date filters require YYYY-MM-DD strings; this cell normalizes inputs defensively.
#   - If your Events DB uses a different date property name, update EVENT_DATE_PROP below.
#

from __future__ import annotations
from typing import Any, Dict, List, Optional
from datetime import datetime, timedelta

events_db_id = RESOLVED_DB["EVENTS_DB"]["database_id"]
events_ds_id = RESOLVED_DB["EVENTS_DB"]["data_source_id"]

# ---- Configure property names (edit here if your Notion names differ) ----
EVENT_TITLE_PROP = "Name"
EVENT_DATE_PROP = "Date"
EVENT_DESC_CANDIDATES = ["Description", "Summary", "Notes"]
EVENT_CATEGORY_PROP = "Event Type"
EVENT_TAGS_PROP = "Target"

# ---- Normalize date inputs to YYYY-MM-DD ----
def _to_yyyy_mm_dd(x: Any) -> Optional[str]:
    if x is None:
        return None
    if isinstance(x, str):
        s = x.strip()
        if len(s) >= 10 and s[4] == "-" and s[7] == "-":
            return s[:10]
        return None
    if isinstance(x, datetime):
        return x.strftime("%Y-%m-%d")
    return None

week_start = _to_yyyy_mm_dd(globals().get("digest_week_start"))
week_end = _to_yyyy_mm_dd(globals().get("digest_week_end"))
fallback_start = _to_yyyy_mm_dd(globals().get("lookback_date_str")) or _to_yyyy_mm_dd(globals().get("lookback_date"))

# If we only have one of week_start/week_end, treat it as a partial window
# (Notion needs explicit operators, so we build what we can)
filters: List[Dict[str, Any]] = []
if week_start:
    filters.append({"property": EVENT_DATE_PROP, "date": {"on_or_after": week_start}})
if week_end:
    filters.append({"property": EVENT_DATE_PROP, "date": {"on_or_before": week_end}})

if not filters:
    if not fallback_start:
        # Last resort: 7 days back from now
        fallback_start = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
    filters.append({"property": EVENT_DATE_PROP, "date": {"on_or_after": fallback_start}})
    print(f"[Cell 10] NOTE: digest week dates missing; using fallback_start={fallback_start}")
else:
    print(f"[Cell 10] Using digest window: start={week_start} end={week_end}")

# Build query payload
query_payload: Dict[str, Any] = {
    "filter": {"and": filters} if len(filters) > 1 else filters[0],
    "sorts": [{"property": EVENT_DATE_PROP, "direction": "descending"}],
    "page_size": 100,
}

# ---- Helpers to read Notion page properties safely ----
def _get_prop(page: Dict[str, Any], prop_name: str) -> Dict[str, Any]:
    return (page.get("properties", {}) or {}).get(prop_name, {}) or {}

def _read_title(page: Dict[str, Any], prop_name: str = EVENT_TITLE_PROP) -> str:
    p = _get_prop(page, prop_name)
    if p.get("type") != "title":
        return ""
    parts = p.get("title", []) or []
    return "".join([t.get("plain_text", "") for t in parts]).strip()

def _read_rich_text(page: Dict[str, Any], prop_name: str) -> str:
    p = _get_prop(page, prop_name)
    if p.get("type") != "rich_text":
        return ""
    parts = p.get("rich_text", []) or []
    return "".join([t.get("plain_text", "") for t in parts]).strip()

def _read_date_start(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    p = _get_prop(page, prop_name)
    if p.get("type") != "date":
        return None
    d = p.get("date") or {}
    return d.get("start")

def _read_select_name(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    p = _get_prop(page, prop_name)
    t = p.get("type")
    if t == "select":
        sel = p.get("select") or {}
        return sel.get("name")
    if t == "status":
        st = p.get("status") or {}
        return st.get("name")
    return None

def _read_multi_select(page: Dict[str, Any], prop_name: str) -> List[str]:
    p = _get_prop(page, prop_name)
    if p.get("type") != "multi_select":
        return []
    items = p.get("multi_select", []) or []
    return [it.get("name") for it in items if it.get("name")]

# ---- Paginate through all events ----
all_events: List[Dict[str, Any]] = []
has_more = True
start_cursor = None

while has_more:
    payload = dict(query_payload)
    if start_cursor:
        payload["start_cursor"] = start_cursor

    resp = notion_request_json("POST", f"data_sources/{events_ds_id}/query", json_body=payload)
    batch = resp.get("results", []) or []
    all_events.extend(batch)

    has_more = bool(resp.get("has_more"))
    start_cursor = resp.get("next_cursor")

print(f"[Cell 10] Weekly events loaded: {len(all_events)}")

# ---- Extract structured event data for downstream processing ----
event_records: List[Dict[str, Any]] = []

for event_page in all_events:
    event_id = event_page.get("id")
    event_title = _read_title(event_page, EVENT_TITLE_PROP)
    event_date = _read_date_start(event_page, EVENT_DATE_PROP)

    # best-effort description
    event_description = ""
    for c in EVENT_DESC_CANDIDATES:
        event_description = _read_rich_text(event_page, c)
        if event_description:
            break

    event_category = _read_select_name(event_page, EVENT_CATEGORY_PROP)
    event_tags = _read_multi_select(event_page, EVENT_TAGS_PROP)

    event_records.append(
        {
            "id": event_id,
            "title": event_title,
            "date": event_date,
            "description": event_description,
            "category": event_category,
            "tags": event_tags,
            "page": event_page,  # keep full page for later evidence linking
        }
    )

# ---- Print compact preview (up to 30) ----
for e in event_records[:30]:
    tags_str = ", ".join(e["tags"]) if e["tags"] else "None"
    print(f"  - {e['title'][:90]}")
    print(f"      Date: {e['date']} | Category: {e['category']} | Tags: {tags_str}")

if len(event_records) > 30:
    print(f"  ... ({len(event_records) - 30} more)")

print(f"\n[Cell 10] Total weekly events cached: {len(event_records)}")


[Cell 10] Using digest window: start=2026-02-02 end=2026-02-08
[Cell 10] Weekly events loaded: 54
  - Sam Altman says Anthropic's Super Bowl spot is 'dishonest' about ChatGPT ads, but he agree
      Date: 2026-02-05 | Category: PEOPLE | Tags: None
  - Sam Altman criticizes rival over Super Bowl ad controversy
      Date: 2026-02-05 | Category: PEOPLE | Tags: None
  - Sam Altman Is Spiraling
      Date: 2026-02-05 | Category: PEOPLE | Tags: None
  - MHRA updates guidance for semaglutide prescribers and patients
      Date: 2026-02-05 | Category: POLICY | Tags: None
  - Public advised to stop using some non-sterile alcohol-free wipes
      Date: 2026-02-05 | Category: POLICY | Tags: None
  - 5 February 2026 £36.7 million to deliver net zero energy networks for consumers Innovate U
      Date: 2026-02-05 | Category: POLICY | Tags: None
  - NIH Launches New Central Resource to Support Replication and Reproducibility
      Date: 2026-02-05 | Category: POLICY | Tags: None
  - CNBC Daily Open

In [35]:
# ============================================================
# Cell 11 — LLM-screen evidence and propose updates WITH quoting current RQ fields (evidence_refs enforced)
# ============================================================
# Overview:
#   For each target RQ:
#     0) Quote CURRENT RQ fields as reference (title, rationale, approach, gap, linked papers)
#     1) LLM screens this week's evidence (papers/events) and selects relevant items
#     2) LLM may propose MULTIPLE update items across categories:
#        - "RQ"          : UPDATED RQ wording/title
#        - "LinkedPaper" : papers to add to Linked Paper relation
#        - "Rationale"   : UPDATED Rationale text (overwrite-ready)
#        - "Approach"    : UPDATED Proposed Approach text (overwrite-ready)
#        - "Gap"         : UPDATED Gap Identified text
#        - "NoChange"    : no meaningful update
#     3) EACH item becomes ONE page in WEEKLY_RQ_UPDATE_DB
#
#   CRITICAL TRACEABILITY REQUIREMENT:
#     - Every non-NoChange item MUST include evidence_refs referencing which P#/E# justify it.
#     - We convert evidence_refs into per-item relevant_papers/relevant_events (Notion page IDs),
#       so each resulting WEEKLY_RQ_UPDATE_DB page can be linked to the exact evidence via relations.
#
# Notion mapping:
#   - Category       : one of ["RQ","LinkedPaper","Rationale","Approach","Gap","NoChange"]
#   - Open Questions : UPDATED FINAL TEXT in Japanese (what will be stored)
#   - Update Summary : WHY / reason + what changed vs current (cite P#/E#)
#   - Evidence Papers (relation) / Evidence Events (relation): per-item relations
#
# Inputs / Outputs:
#   Inputs:
#     - rq_records (Cell 08)          # each includes rq["page"] (full Notion page object)
#     - paper_records (Cell 09)       # includes LIT_DB fields: core_idea/findings/notes/methods/datasets
#     - event_records (Cell 10)
#     - openai_client, llm_model, llm_temperature
#   Outputs:
#     - rq_updates: list[dict]  # one dict = one future Notion page item, with per-item evidence relations
#
# Notes:
#   - The same evidence can justify multiple categories (multiple items).
#   - If a selected paper is NOT currently linked, a "LinkedPaper" item MUST be created for it.
#   - We refuse to accept non-NoChange items without evidence_refs; they are downgraded to NoChange with warning text.
#

from __future__ import annotations
from typing import Any, Dict, List, Tuple, Optional, Set
import json

MAX_PAPERS_IN_PROMPT = 12
MAX_EVENTS_IN_PROMPT = 12

# ---- RQ_DB property names (adjust if your Notion names differ) ----
RQ_TITLE_PROP = "Name"
RQ_LINKED_PAPER_PROP = "Linked Paper"
RQ_RATIONALE_PROP = "Rationale / Background"
RQ_APPROACH_PROP = "Proposed Approach"
RQ_GAP_PROP = "Gap Identified"

# ------------------------------------------------------------------
# Utility helpers
# ------------------------------------------------------------------
def _clip(s: str, n: int) -> str:
    s = (s or "").strip()
    if len(s) <= n:
        return s
    return s[:n].rstrip() + "…"

def _tokenize(s: str) -> List[str]:
    s = (s or "").lower()
    for ch in "\n\t,.;:()[]{}<>/\\|!?\"'":
        s = s.replace(ch, " ")
    return [t for t in s.split(" ") if t]

def _paper_text_for_scoring(p: Dict[str, Any]) -> str:
    parts = [
        p.get("title", "") or "",
        p.get("summary", "") or "",
        p.get("core_idea", "") or "",
        p.get("findings", "") or "",
        p.get("notes", "") or "",
        p.get("methods", "") or "",
        p.get("datasets", "") or "",
        " ".join(p.get("tags") or []),
    ]
    return " ".join([x for x in parts if x]).strip()

def _cheap_relevance_score(rq: Dict[str, Any], item: Dict[str, Any]) -> int:
    rq_tags = set([t.lower() for t in (rq.get("tags") or []) if t])
    it_tags = set([t.lower() for t in (item.get("tags") or []) if t])
    tag_overlap = len(rq_tags & it_tags)

    rq_text = " ".join([
        rq.get("title", ""),
        rq.get("description", ""),
        " ".join(rq.get("tags") or []),
    ]).lower()

    if "findings" in item or "core_idea" in item:
        it_text = _paper_text_for_scoring(item).lower()
    else:
        it_text = " ".join([
            item.get("title", ""),
            item.get("summary", ""),
            item.get("description", ""),
            " ".join(item.get("tags") or []),
        ]).lower()

    kw_overlap = len(set(_tokenize(rq_text)) & set(_tokenize(it_text)))
    return tag_overlap * 3 + min(kw_overlap, 18)

def _pick_top(items: List[Dict[str, Any]], k: int, score_fn) -> List[Dict[str, Any]]:
    scored = [(score_fn(it), it) for it in items]
    scored.sort(key=lambda x: x[0], reverse=True)
    positives = [it for s, it in scored[:k] if s > 0]
    return positives if positives else [it for s, it in scored[:k]]

def _build_evidence_block(papers: List[Dict[str, Any]], events: List[Dict[str, Any]]) -> str:
    lines: List[str] = []

    lines.append("# Evidence Papers (candidates)")
    if papers:
        for i, p in enumerate(papers, 1):
            lines.append(f"P{i}. {p.get('title','')}")
            if p.get("authors_year"):
                lines.append(f"    Authors&Year: {_clip(p.get('authors_year',''), 200)}")
            if p.get("decision"):
                lines.append(f"    Decision: {p.get('decision')}")
            if p.get("tags"):
                lines.append(f"    Tags: {', '.join(p.get('tags') or [])}")

            # richer LIT_DB fields
            if p.get("core_idea"):
                lines.append(f"    Core Idea: {_clip(p.get('core_idea',''), 700)}")
            if p.get("findings"):
                lines.append(f"    Findings: {_clip(p.get('findings',''), 900)}")
            if p.get("methods"):
                lines.append(f"    Methods: {_clip(p.get('methods',''), 700)}")
            if p.get("datasets"):
                lines.append(f"    Datasets: {_clip(p.get('datasets',''), 500)}")
            if p.get("notes"):
                lines.append(f"    Notes: {_clip(p.get('notes',''), 700)}")

            if p.get("summary"):
                lines.append(f"    Summary(fallback): {_clip(p.get('summary',''), 400)}")

            lines.append(f"    NotionPageID: {p.get('id')}")
            lines.append("")
    else:
        lines.append("None\n")

    lines.append("# Evidence Events (candidates)")
    if events:
        for i, e in enumerate(events, 1):
            lines.append(f"E{i}. {e.get('title','')}")
            if e.get("date"):
                lines.append(f"    Date: {e.get('date')}")
            if e.get("category"):
                lines.append(f"    Category: {e.get('category')}")
            if e.get("description"):
                lines.append(f"    Description: {_clip(e.get('description',''), 800)}")
            if e.get("tags"):
                lines.append(f"    Tags: {', '.join(e.get('tags') or [])}")
            lines.append(f"    NotionPageID: {e.get('id')}")
            lines.append("")
    else:
        lines.append("None\n")

    return "\n".join(lines)

# ---- Read current RQ fields from rq["page"] ----
def _prop(page: Dict[str, Any], name: str) -> Dict[str, Any]:
    return (page.get("properties", {}) or {}).get(name, {}) or {}

def _read_title(page: Dict[str, Any], name: str) -> str:
    p = _prop(page, name)
    if p.get("type") != "title":
        return ""
    return "".join(t.get("plain_text", "") for t in (p.get("title") or [])).strip()

def _read_rich_text(page: Dict[str, Any], name: str) -> str:
    p = _prop(page, name)
    if p.get("type") != "rich_text":
        return ""
    return "".join(t.get("plain_text", "") for t in (p.get("rich_text") or [])).strip()

def _read_relation_ids(page: Dict[str, Any], name: str) -> List[str]:
    p = _prop(page, name)
    if p.get("type") != "relation":
        return []
    rel = p.get("relation") or []
    out: List[str] = []
    for r in rel:
        if isinstance(r, dict) and r.get("id"):
            out.append(r["id"])
    return out

# ---- Evidence refs parsing ----
def _split_refs(refs: Any) -> Tuple[List[str], List[str]]:
    """
    refs: list like ["P1","E2"] or string like "P1, E2"
    returns (paper_indices, event_indices)
    """
    if refs is None:
        return ([], [])
    items: List[str] = []
    if isinstance(refs, list):
        for x in refs:
            if isinstance(x, str):
                items.append(x.strip())
    elif isinstance(refs, str):
        # split by comma/space
        parts = [p.strip() for p in refs.replace("\n", ",").split(",")]
        items.extend([p for p in parts if p])
    else:
        return ([], [])

    pidx: List[str] = []
    eidx: List[str] = []
    for it in items:
        it2 = it.strip()
        if it2.startswith("P"):
            pidx.append(it2)
        elif it2.startswith("E"):
            eidx.append(it2)
    return (pidx, eidx)

def _dedup_keep_order(xs: List[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in xs:
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out

# ------------------------------------------------------------------
# Main loop
# ------------------------------------------------------------------
rq_updates: List[Dict[str, Any]] = []

for rq in rq_records:
    rq_id = rq["id"]
    rq_tags = rq.get("tags", []) or []
    rq_page = rq.get("page") or {}

    current_rq_title = _read_title(rq_page, RQ_TITLE_PROP) or rq.get("title", "")
    current_rationale = _read_rich_text(rq_page, RQ_RATIONALE_PROP)
    current_approach = _read_rich_text(rq_page, RQ_APPROACH_PROP)
    current_gap = _read_rich_text(rq_page, RQ_GAP_PROP)
    current_linked_paper_ids = _read_relation_ids(rq_page, RQ_LINKED_PAPER_PROP)

    # Evidence candidates (score uses richer paper fields now)
    pre_papers = _pick_top(paper_records, MAX_PAPERS_IN_PROMPT, lambda it: _cheap_relevance_score(rq, it))
    pre_events = _pick_top(event_records, MAX_EVENTS_IN_PROMPT, lambda it: _cheap_relevance_score(rq, it))
    evidence_block = _build_evidence_block(pre_papers, pre_events)

    print(f"\n[Cell 11] LLM proposing updates (evidence_refs required) for RQ: {current_rq_title}")
    print(f"  Candidate papers={len(pre_papers)} | events={len(pre_events)}")

    prompt = f"""
You are a research assistant maintaining a Research Question (RQ) knowledge base.

# Categories (use EXACTLY these)
- RQ
- LinkedPaper
- Rationale
- Approach
- Gap
- NoChange

IMPORTANT RULES:
- The SAME paper/event MAY justify MULTIPLE categories.
- If you select a paper NOT in Current Linked Paper IDs, you MUST create a "LinkedPaper" item for it.
- Do NOT merge multiple categories into one item. One item = one category.
- Every non-NoChange item MUST include evidence_refs that point to the exact evidence indices (P#/E#).
- evidence_refs must be a non-empty list like ["P1","E2"] for non-NoChange items.

Guidance:
- Use Core Idea / Findings / Notes mainly for Rationale updates.
- Use Methods / Datasets for Approach updates.
- Use dataset limits, assumptions, missing validations for Gap updates.
- LinkedPaper: add papers that are central evidence AND not already linked.

# CURRENT RQ (reference)
Title:
{current_rq_title}

Tags:
{", ".join(rq_tags) if rq_tags else "None"}

Current Rationale:
{current_rationale if current_rationale else "(empty)"}

Current Proposed Approach:
{current_approach if current_approach else "(empty)"}

Current Gap Identified:
{current_gap if current_gap else "(empty)"}

Current Linked Paper IDs:
{", ".join(current_linked_paper_ids) if current_linked_paper_ids else "(none)"}

{evidence_block}

# TASK
Step 1) Select up to 5 papers and 5 events most relevant to this RQ.
- Return indices as selected_papers (e.g., ["P1","P3"]) and selected_events (e.g., ["E2"]).

Step 2) Propose ZERO, ONE, or MULTIPLE items across categories.
- If no meaningful updates exist, return ONE item with category "NoChange" and empty evidence_refs.

For EACH item:
- category: one of the categories above
- updated_text_ja: FINAL Japanese text to store (overwrite-ready)
- reason: WHY this update is warranted and what changed vs current (cite P#/E#)
+ reason: 
+   Explain WHY this update is warranted, using the ACTUAL paper/event content.
+   The reason MUST:
+   - Explicitly mention the paper/event title(s)
+   - Briefly summarize WHAT the paper/event shows (methods, findings, or core idea)
+   - Explain HOW this changes or improves the CURRENT RQ field (Rationale / Approach / Gap)
+   - Cite evidence indices (P#/E#) at the end
- confidence: float 0..1
- linked_paper_additions: list of NotionPageID strings (ONLY for LinkedPaper; otherwise [])
+ BAD example (DO NOT DO THIS):
+ - "P1の方法論が有用であるため。P1を引用。"
+
+ GOOD example:
+ - "Smith et al. (2023)『Corporate Venture Capital as Narrative Infrastructure』は、
+    CVCが資金提供者ではなくエコシステム内の正統化装置として機能することを、
+    欧米CVCの比較分析によって示している。
+    現行Rationaleでは制度設計に焦点が当たっているため、
+    この物語・正統化の視点を追加する必要がある（P1）。"


# OUTPUT (STRICT JSON ONLY)
{{
  "selected_papers": ["P1"],
  "selected_events": ["E1"],
  "items": [
    {{
      "category": "Rationale",
      "evidence_refs": ["P1","E1"],
      "updated_text_ja": "（変更後の最終Rationale本文）",
      "reason": "（なぜ更新が必要か＋現行からの差分。P#/E#引用）",
      "confidence": 0.7,
      "linked_paper_additions": []
    }}
  ]
}}
""".strip()

    try:
        resp = openai_client.chat.completions.create(
            model=llm_model,
            messages=[
                {"role": "system", "content": "Return STRICT JSON only. No markdown."},
                {"role": "user", "content": prompt},
            ],
            temperature=llm_temperature,
            max_tokens=1700,
        )
        raw = resp.choices[0].message.content.strip()
        data = json.loads(raw)

        # index mapping
        idx_to_paper = {f"P{i}": p for i, p in enumerate(pre_papers, 1)}
        idx_to_event = {f"E{i}": e for i, e in enumerate(pre_events, 1)}

        # global selection (still useful)
        selected_paper_ids_global = [idx_to_paper[i]["id"] for i in data.get("selected_papers", []) if i in idx_to_paper]
        selected_event_ids_global = [idx_to_event[i]["id"] for i in data.get("selected_events", []) if i in idx_to_event]

        items = data.get("items") or []
        if not items:
            items = [{
                "category": "NoChange",
                "evidence_refs": [],
                "updated_text_ja": "今週は更新なし。",
                "reason": "新規の決定的Evidenceがなかったため。",
                "confidence": 0.0,
                "linked_paper_additions": [],
            }]

        # ---- enforce per-item evidence_refs ----
        coerced_items: List[Dict[str, Any]] = []
        for it in items:
            cat = (it.get("category") or "NoChange").strip()
            ev_refs = it.get("evidence_refs", None)

            # Normalize to list of strings
            if isinstance(ev_refs, str):
                # allow comma separated
                ev_refs_norm = [x.strip() for x in ev_refs.replace("\n", ",").split(",") if x.strip()]
            elif isinstance(ev_refs, list):
                ev_refs_norm = [x.strip() for x in ev_refs if isinstance(x, str) and x.strip()]
            else:
                ev_refs_norm = []

            # If non-NoChange but no evidence_refs, downgrade to NoChange
            if cat != "NoChange" and len(ev_refs_norm) == 0:
                coerced_items.append({
                    "category": "NoChange",
                    "evidence_refs": [],
                    "updated_text_ja": "今週は更新なし（Evidence参照が明示されなかったため提案を棄却）。",
                    "reason": "LLM提案が evidence_refs を満たさないため（traceability contract）。",
                    "confidence": 0.0,
                    "linked_paper_additions": [],
                })
                continue

            it2 = dict(it)
            it2["category"] = cat
            it2["evidence_refs"] = ev_refs_norm
            coerced_items.append(it2)

        # ---- build per-item records with per-item evidence IDs ----
        for it in coerced_items:
            category = (it.get("category") or "NoChange").strip()
            updated_text_ja = (it.get("updated_text_ja") or "").strip()
            reason = (it.get("reason") or "").strip()
            confidence = float(it.get("confidence") or 0.0)

            # Per-item evidence refs -> per-item IDs
            pidx, eidx = _split_refs(it.get("evidence_refs"))
            per_item_paper_ids = [idx_to_paper[x]["id"] for x in pidx if x in idx_to_paper]
            per_item_event_ids = [idx_to_event[x]["id"] for x in eidx if x in idx_to_event]

            per_item_paper_ids = _dedup_keep_order(per_item_paper_ids)
            per_item_event_ids = _dedup_keep_order(per_item_event_ids)

            # For LinkedPaper, enforce additions are subset of referenced papers (best-effort)
            linked_add = it.get("linked_paper_additions") or []
            if not isinstance(linked_add, list):
                linked_add = []
            linked_add = [x for x in linked_add if isinstance(x, str) and x.strip()]
            linked_add = _dedup_keep_order(linked_add)

            if category == "LinkedPaper":
                # If LLM didn't fill linked_paper_additions, infer from referenced papers not already linked
                if not linked_add:
                    linked_add = [pid for pid in per_item_paper_ids if pid and (pid not in current_linked_paper_ids)]
                # If it added papers that are not referenced, keep but warn in reason (do not drop silently)
                not_ref = [pid for pid in linked_add if pid not in per_item_paper_ids]
                if not_ref:
                    reason = (reason + "\n\n[WARN] linked_paper_additions contains papers not listed in evidence_refs.").strip()

            rq_updates.append({
                "rq_id": rq_id,
                "rq_title": current_rq_title,

                # Notion fields
                "open_questions": [updated_text_ja] if updated_text_ja else [],
                "update_summary": reason,
                "confidence": confidence,

                # IMPORTANT: per-item evidence (traceable relations)
                "relevant_papers": per_item_paper_ids,
                "relevant_events": per_item_event_ids,
                "linked_paper_additions": linked_add,

                # Category
                "categories": [category],

                # Stats/debug
                "evidence_count": len(per_item_paper_ids) + len(per_item_event_ids),
                "debug_raw_json": raw[:4000],
            })

        cats = [it.get("category") for it in coerced_items]
        print(f"  ✓ done | pages={len(coerced_items)} | categories={cats}")
        # quick traceability preview
        for it in coerced_items[:6]:
            print(f"    - {it.get('category')}: evidence_refs={it.get('evidence_refs')}")

    except Exception as e:
        print(f"  ERROR: {str(e)}")
        rq_updates.append({
            "rq_id": rq_id,
            "rq_title": current_rq_title,
            "open_questions": ["今週は更新生成に失敗しました。"],
            "update_summary": f"Error: {str(e)}",
            "confidence": 0.0,
            "relevant_papers": [],
            "relevant_events": [],
            "linked_paper_additions": [],
            "categories": ["NoChange"],
            "evidence_count": 0,
            "debug_raw_json": "",
        })

print(f"\n[Cell 11] Total per-category update pages generated (traceable): {len(rq_updates)}")



[Cell 11] LLM proposing updates (evidence_refs required) for RQ: [v1.1] スタートアップ投資バブルを長期的なイノベーションの軌道に変換するための神話や物語構造と、CVC戦略の役割は何か？
  Candidate papers=7 | events=12
  ✓ done | pages=1 | categories=['Rationale']
    - Rationale: evidence_refs=['P1', 'E1']

[Cell 11] LLM proposing updates (evidence_refs required) for RQ: [v1.1] 外国系VCとコーポレートVCは、新興市場におけるスタートアップの成長とイノベーション成果にどのように異なる影響を与えるか？
  Candidate papers=7 | events=12
  ✓ done | pages=3 | categories=['Rationale', 'Approach', 'Gap']
    - Rationale: evidence_refs=['P1']
    - Approach: evidence_refs=['P2']
    - Gap: evidence_refs=['P3']

[Cell 11] LLM proposing updates (evidence_refs required) for RQ: LP投資者として行動する政府系ファンド（SWF）は、国内スタートアップ・エコシステムの形成にどのような影響を与えるか？
  Candidate papers=7 | events=12
  ✓ done | pages=3 | categories=['Rationale', 'Approach', 'Gap']
    - Rationale: evidence_refs=['P1']
    - Approach: evidence_refs=['P2']
    - Gap: evidence_refs=['P3']

[Cell 11] LLM proposing updates (evidence_refs required) for RQ: スタートアップ投資バ

In [37]:
# ============================================================
# Cell 12 — Normalize per-category proposals (Open Questions), confidence, and per-item evidence (traceability)
# ============================================================
# Overview:
#   Normalize outputs from Cell 11 where each entry corresponds to ONE category item
#   (i.e., one future WEEKLY_RQ_UPDATE_DB page).
#
#   Display / storage contract (per your request):
#     - Open Questions field will store Japanese "after-change" FINAL text to store (overwrite-ready)
#     - Update Summary field will store the rationale / reason, citing evidence indices (P#/E#) when applicable
#
#   Traceability contract (NEW / important):
#     - Each non-NoChange item SHOULD have at least one evidence relation:
#         relevant_papers and/or relevant_events
#     - This cell does NOT invent evidence; it only validates and warns.
#
# Inputs / Outputs:
#   Inputs:
#     - rq_updates (list[dict]) from Cell 11 (per-category items; already per-item evidence)
#   Outputs:
#     - rq_updates_norm (list[dict]) normalized items with:
#         - open_questions_norm (list[str])
#         - update_summary_norm (str)
#         - confidence_norm (float)
#         - categories_norm (list[str])  # exactly 1 element
#         - category_text (str)
#         - relevant_papers_norm (list[str])
#         - relevant_events_norm (list[str])
#         - linked_paper_additions_norm (list[str])
#         - has_any_change (bool)
#         - traceability_ok (bool)   # non-NoChange must have evidence
#         - traceability_warn (str)  # warning text if any
#
# Notes:
#   - This cell does NOT write to Notion.
#   - If category is missing, defaults to "NoChange".
#   - If Open Questions is empty for non-NoChange, we keep the item but warn.
#   - If evidence is empty for non-NoChange, we keep the item but warn (should be rare because Cell 11 enforces evidence_refs).
#

from __future__ import annotations
from typing import Any, Dict, List, Optional, Set

VALID_CATEGORIES = {"RQ", "LinkedPaper", "Rationale", "Approach", "Gap", "NoChange"}

def _dedup_keep_order(xs: List[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in xs:
        if not isinstance(x, str):
            continue
        s = x.strip()
        if not s or s in seen:
            continue
        seen.add(s)
        out.append(s)
    return out

def _normalize_open_questions(x: Any) -> List[str]:
    """
    Accept list[str] or a single string. Return clean list[str].
    For this notebook, Open Questions is used to store Japanese proposal text (final overwrite-ready text).
    """
    if x is None:
        return []
    if isinstance(x, list):
        out: List[str] = []
        for it in x:
            if isinstance(it, str):
                s = it.strip()
                s = s.lstrip("-•").strip()
                if s:
                    out.append(s)
        return out
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        lines = [ln.strip().lstrip("-•").strip() for ln in s.split("\n") if ln.strip()]
        return [ln for ln in lines if ln]
    return []

def _normalize_update_summary(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    # allow accidental list (join)
    if isinstance(x, list):
        parts = [p.strip() for p in x if isinstance(p, str) and p.strip()]
        return "\n".join(parts).strip()
    return ""

def _clamp01(v: float) -> float:
    return 0.0 if v < 0 else 1.0 if v > 1 else v

def _normalize_confidence(x: Any) -> float:
    try:
        if x is None:
            return 0.0
        if isinstance(x, (int, float)):
            return _clamp01(float(x))
        if isinstance(x, str):
            s = x.strip()
            if not s:
                return 0.0
            return _clamp01(float(s))
    except Exception:
        return 0.0
    return 0.0

def _normalize_single_category(x: Any) -> str:
    """
    Expect exactly ONE category per item.
    Accept list or string; return a single category string.
    """
    if x is None:
        return "NoChange"
    if isinstance(x, list):
        cats = [c.strip() for c in x if isinstance(c, str) and c.strip()]
        if not cats:
            return "NoChange"
        return cats[0]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return "NoChange"
        # if comma-separated, take first
        first = s.split(",")[0].strip()
        return first or "NoChange"
    return "NoChange"

def _normalize_ids_list(x: Any) -> List[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return _dedup_keep_order([v for v in x if isinstance(v, str)])
    if isinstance(x, str):
        # allow comma-separated
        parts = [p.strip() for p in x.replace("\n", ",").split(",") if p.strip()]
        return _dedup_keep_order(parts)
    return []

def _has_any_change(category: str, upd: Dict[str, Any], open_q: List[str]) -> bool:
    if category != "NoChange":
        return True
    if open_q:
        return True
    if (upd.get("update_summary") or "").strip():
        return True
    if upd.get("linked_paper_additions"):
        return True
    if upd.get("relevant_papers") or upd.get("relevant_events"):
        return True
    return False

def _traceability_ok(category: str, papers: List[str], events: List[str]) -> Tuple[bool, str]:
    if category == "NoChange":
        return True, ""
    if papers or events:
        return True, ""
    return False, "Non-NoChange item has no evidence relations (relevant_papers/events are empty)."

rq_updates_norm: List[Dict[str, Any]] = []

for upd in rq_updates:
    category = _normalize_single_category(upd.get("categories"))
    if category not in VALID_CATEGORIES:
        # coerce unknown categories to NoChange to keep schema stable
        category = "NoChange"

    open_q = _normalize_open_questions(upd.get("open_questions"))
    summary = _normalize_update_summary(upd.get("update_summary"))
    conf = _normalize_confidence(upd.get("confidence"))

    papers = _normalize_ids_list(upd.get("relevant_papers"))
    events = _normalize_ids_list(upd.get("relevant_events"))
    linked_add = _normalize_ids_list(upd.get("linked_paper_additions"))

    # Traceability validation
    ok, warn = _traceability_ok(category, papers, events)

    # Content validation warnings (do not drop items)
    content_warns: List[str] = []
    if category != "NoChange" and not open_q:
        content_warns.append("Non-NoChange item has empty Open Questions (no updated_text_ja).")
    if category != "NoChange" and not summary:
        content_warns.append("Non-NoChange item has empty Update Summary (no reason text).")

    traceability_warn = warn
    if content_warns:
        traceability_warn = (traceability_warn + "\n" if traceability_warn else "") + "\n".join(content_warns)

    norm = dict(upd)
    norm["open_questions_norm"] = open_q
    norm["update_summary_norm"] = summary
    norm["confidence_norm"] = conf
    norm["categories_norm"] = [category]
    norm["category_text"] = category

    norm["relevant_papers_norm"] = papers
    norm["relevant_events_norm"] = events
    norm["linked_paper_additions_norm"] = linked_add

    norm["has_any_change"] = _has_any_change(category, upd, open_q)
    norm["traceability_ok"] = ok
    norm["traceability_warn"] = traceability_warn

    rq_updates_norm.append(norm)

# ---- Summary diagnostics ----
n_total = len(rq_updates_norm)
n_nochange = sum(1 for u in rq_updates_norm if u.get("category_text") == "NoChange")
n_trace_bad = sum(1 for u in rq_updates_norm if not u.get("traceability_ok", True))
avg_conf = sum(u.get("confidence_norm", 0.0) for u in rq_updates_norm) / max(n_total, 1)

print("[Cell 12] Normalization complete (per-category items).")
print(f"  total items (pages)        : {n_total}")
print(f"  NoChange items             : {n_nochange}")
print(f"  traceability warnings      : {n_trace_bad}")
print(f"  avg confidence             : {avg_conf:.2f}")

# ---- Compact preview (first 20) ----
for u in rq_updates_norm[:20]:
    oq = u.get("open_questions_norm") or []
    oq_preview = (oq[0][:80] + "…") if oq and len(oq[0]) > 80 else (oq[0] if oq else "")
    papers_n = len(u.get("relevant_papers_norm") or [])
    events_n = len(u.get("relevant_events_norm") or [])
    ok = u.get("traceability_ok", True)
    warn_mark = "" if ok else " [WARN]"
    print(f"  - {u.get('rq_title','')[:55]} | {u.get('category_text')}{warn_mark}")
    print(f"      confidence={u.get('confidence_norm'):.2f} | evidence: {papers_n} papers, {events_n} events | proposal_ja_preview={oq_preview!r}")
    if u.get("traceability_warn"):
        print(f"      warn: {_clip(u.get('traceability_warn'), 160) if ' _clip' in globals() else u.get('traceability_warn')[:160]}")


[Cell 12] Normalization complete (per-category items).
  total items (pages)        : 20
  NoChange items             : 0
  traceability warnings      : 0
  avg confidence             : 0.76
  - [v1.1] スタートアップ投資バブルを長期的なイノベーションの軌道に変換するための神話や物語構造と、CVC戦 | Rationale
      confidence=0.80 | evidence: 1 papers, 1 events | proposal_ja_preview='スタートアップ投資バブルを長期的なイノベーションの軌道に変換するには、どのような神話や物語構造が必要か？最新のJVCAパネル討論によると、CVCの戦略進化や投資…'
  - [v1.1] 外国系VCとコーポレートVCは、新興市場におけるスタートアップの成長とイノベーション成果にどのよ | Rationale
      confidence=0.80 | evidence: 1 papers, 0 events | proposal_ja_preview='本研究は、外国系VCとコーポレートVCが新興市場におけるスタートアップの成長とイノベーション成果に与える影響を、歴史的な視点から分析することの重要性を示している…'
  - [v1.1] 外国系VCとコーポレートVCは、新興市場におけるスタートアップの成長とイノベーション成果にどのよ | Approach
      confidence=0.75 | evidence: 1 papers, 0 events | proposal_ja_preview='本研究では、企業レベルのパネルデータを用いて、外国系VCとCVCが支援するスタートアップの成長、イノベーション成果、及び出口戦略に及ぼす影響を比較する。特に、元…'
  - [v1.1] 外国系VCとコーポレートVCは、新興市場におけるスタートアップの成長とイノベーション成果にどのよ | Gap
      confidence=0.70 | evidence: 1 papers, 0 event

In [38]:
# ============================================================
# Cell 13 — Link evidence papers and events to each per-category RQ update item
# ============================================================
# Overview:
#   Prepare per-item relation payloads so that each future WEEKLY_RQ_UPDATE_DB page
#   will be traceably linked to the exact evidence used:
#     - Evidence Papers (relation)  -> PAPERS_DB
#     - Evidence Events (relation)  -> EVENTS_DB
#   Also prepares "Research Question" relation (to RQ_DB) and optional "Week" relation
#   (to WEEKLY_DIGESTS_DB) and optional "Related Theme(s)" (if available later).
#
# Inputs / Outputs:
#   Inputs:
#     - rq_updates_norm from Cell 12 (per-item; has relevant_papers_norm/relevant_events_norm)
#     - weekly_digest_id from Cell 07
#     - RESOLVED_DB dict from Cell 03 (database IDs) [for reference only]
#     - weekly_rq_update_prop_types from Cell 04 (optional; if present we validate prop names)
#   Outputs:
#     - rq_updates_linked: list[dict] each item includes:
#         - evidence_papers_rel (list[{"id":...}])
#         - evidence_events_rel (list[{"id":...}])
#         - rq_rel            (list[{"id":...}])
#         - week_rel          (list[{"id":...}])  # if property exists
#         - category_text     (str)
#         - open_questions_norm (list[str])
#         - update_summary_norm (str)
#         - confidence_norm (float)
#
# Notes:
#   - We DO NOT write to Notion here. Cell 14 does the actual page creation.
#   - We intentionally link evidence per item (not per RQ) to preserve auditability.
#   - Property names below assume your WEEKLY_RQ_UPDATE_DB schema:
#       - "Evidence Papers" (relation) -> PAPERS_DB
#       - "Evidence Events" (relation) -> EVENTS_DB
#       - "Research Question" (relation) -> RQ_DB
#       - "Week" (relation) -> WEEKLY_DIGESTS_DB
#     If your Notion uses different names, edit the constants.
#

from __future__ import annotations
from typing import Any, Dict, List, Optional, Set

# ---- WEEKLY_RQ_UPDATE_DB property names (edit if your Notion names differ) ----
WUP_TITLE_PROP = "Name"
WUP_WEEK_PROP = "Week"  # relation to WEEKLY_DIGESTS_DB
WUP_RQ_PROP = "Research Question"  # relation to RQ_DB
WUP_CATEGORY_PROP = "Category"  # rich_text (per your note)
WUP_UPDATE_SUMMARY_PROP = "Update Summary"  # rich_text
WUP_OPEN_QUESTIONS_PROP = "Open Questions"  # rich_text
WUP_CONFIDENCE_PROP = "Confidence"  # number
WUP_STATUS_PROP = "Status"  # select
WUP_EVIDENCE_EVENTS_PROP = "Evidence Events"  # relation -> EVENTS_DB
WUP_EVIDENCE_PAPERS_PROP = "Evidence Papers"  # relation -> PAPERS_DB
WUP_RELATED_THEMES_PROP = "Related Theme(s)"  # relation -> WEEKLY_THEMES_DB (optional)

# ---- Helpers ----
def _dedup_keep_order(xs: List[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in xs:
        if not isinstance(x, str):
            continue
        s = x.strip()
        if not s or s in seen:
            continue
        seen.add(s)
        out.append(s)
    return out

def _as_relation(ids: List[str]) -> List[Dict[str, str]]:
    ids2 = _dedup_keep_order(ids)
    return [{"id": i} for i in ids2]

def _join_lines(lines: List[str]) -> str:
    # Store as multi-line rich text. Keep bullet markers if user included them.
    clean = []
    for ln in lines or []:
        if not isinstance(ln, str):
            continue
        s = ln.strip()
        if s:
            clean.append(s)
    return "\n".join(clean).strip()

def _safe_get(d: Dict[str, Any], k: str, default=None):
    v = d.get(k, default)
    return default if v is None else v

# ---- Optional: validate property existence against cached types (if available) ----
def _prop_exists(prop_types: Optional[Dict[str, Any]], name: str) -> bool:
    if not prop_types:
        return True
    return name in prop_types

prop_types = globals().get("weekly_rq_update_prop_types", None)  # from Cell 04 if you cached it

# Warn for missing expected props (non-fatal)
expected_props = [
    WUP_TITLE_PROP, WUP_CATEGORY_PROP, WUP_OPEN_QUESTIONS_PROP, WUP_UPDATE_SUMMARY_PROP,
    WUP_CONFIDENCE_PROP, WUP_RQ_PROP, WUP_EVIDENCE_PAPERS_PROP, WUP_EVIDENCE_EVENTS_PROP,
]
optional_props = [WUP_WEEK_PROP, WUP_STATUS_PROP, WUP_RELATED_THEMES_PROP]

missing_expected = [p for p in expected_props if not _prop_exists(prop_types, p)]
missing_optional = [p for p in optional_props if not _prop_exists(prop_types, p)]

if missing_expected:
    print("[Cell 13] WARNING: some expected WEEKLY_RQ_UPDATE_DB properties are not present in cached schema:")
    for p in missing_expected:
        print(f"  - {p}")
if missing_optional:
    print("[Cell 13] NOTE: optional properties not present (OK):")
    for p in missing_optional:
        print(f"  - {p}")

# ---- Build linked items ----
rq_updates_linked: List[Dict[str, Any]] = []

for u in rq_updates_norm:
    category = u.get("category_text") or "NoChange"

    # per-item evidence (already per-item from Cell 11)
    paper_ids = _safe_get(u, "relevant_papers_norm", [])
    event_ids = _safe_get(u, "relevant_events_norm", [])

    evidence_papers_rel = _as_relation(paper_ids)
    evidence_events_rel = _as_relation(event_ids)

    # relations to RQ + Week
    rq_rel = _as_relation([u.get("rq_id")]) if u.get("rq_id") else []
    week_rel = _as_relation([weekly_digest_id]) if "weekly_digest_id" in globals() and weekly_digest_id else []

    # Text fields
    open_q_text = _join_lines(_safe_get(u, "open_questions_norm", []))
    reason_text = (u.get("update_summary_norm") or "").strip()

    # A compact title for the update page
    # Example: "RQ3: Rationale (2026-02-02–2026-02-08)" or "RQ: LinkedPaper"
    rq_title = (u.get("rq_title") or "").strip()
    week_label = ""
    if "digest_week_start" in globals() and "digest_week_end" in globals() and digest_week_start and digest_week_end:
        week_label = f"{digest_week_start}–{digest_week_end}"
    elif "digest_title" in globals() and digest_title:
        week_label = digest_title

    name_title = f"{rq_title}: {category}"
    if week_label:
        name_title += f" ({week_label})"

    # Store everything needed for Cell 14 write
    linked_item = dict(u)
    linked_item.update({
        "name_title": name_title,
        "evidence_papers_rel": evidence_papers_rel,
        "evidence_events_rel": evidence_events_rel,
        "rq_rel": rq_rel,
        "week_rel": week_rel,
        "open_q_text": open_q_text,
        "reason_text": reason_text,
    })

    rq_updates_linked.append(linked_item)

# ---- Diagnostics preview ----
print("[Cell 13] Evidence linkage prepared per item.")
print(f"  items: {len(rq_updates_linked)}")
n_warn = sum(1 for x in rq_updates_linked if (x.get("category_text") != "NoChange") and (len(x.get("evidence_papers_rel", [])) + len(x.get("evidence_events_rel", [])) == 0))
if n_warn:
    print(f"  WARNING: {n_warn} non-NoChange items have 0 evidence relations (should be rare).")

for x in rq_updates_linked[:15]:
    cat = x.get("category_text")
    np = len(x.get("evidence_papers_rel") or [])
    ne = len(x.get("evidence_events_rel") or [])
    print(f"  - {x.get('name_title','')[:90]}")
    print(f"      category={cat} | evidence: {np} papers, {ne} events | confidence={x.get('confidence_norm', 0.0):.2f}")


[Cell 13] Evidence linkage prepared per item.
  items: 20
  - [v1.1] スタートアップ投資バブルを長期的なイノベーションの軌道に変換するための神話や物語構造と、CVC戦略の役割は何か？: Rationale (2026-02-02–202
      category=Rationale | evidence: 1 papers, 1 events | confidence=0.80
  - [v1.1] 外国系VCとコーポレートVCは、新興市場におけるスタートアップの成長とイノベーション成果にどのように異なる影響を与えるか？: Rationale (2026-02-0
      category=Rationale | evidence: 1 papers, 0 events | confidence=0.80
  - [v1.1] 外国系VCとコーポレートVCは、新興市場におけるスタートアップの成長とイノベーション成果にどのように異なる影響を与えるか？: Approach (2026-02-02
      category=Approach | evidence: 1 papers, 0 events | confidence=0.75
  - [v1.1] 外国系VCとコーポレートVCは、新興市場におけるスタートアップの成長とイノベーション成果にどのように異なる影響を与えるか？: Gap (2026-02-02–2026
      category=Gap | evidence: 1 papers, 0 events | confidence=0.70
  - LP投資者として行動する政府系ファンド（SWF）は、国内スタートアップ・エコシステムの形成にどのような影響を与えるか？: Rationale (2026-02-02–2026-02
      category=Rationale | evidence: 1 papers, 0 events | confidence=0.80
  - LP投資者として行動する政府系ファンド（SWF）は、国内スタートアップ・エコシステムの形成にどのような影響を与えるか？: Approach (2026-02-02–2026-02-
      cat

In [39]:
# ============================================================
# Cell 14 — Write per-category RQ update records to WEEKLY_RQ_UPDATE_DB
# ============================================================
# Overview:
#   Create ONE Notion page per per-category RQ update item (rq_updates_linked).
#   Each page links:
#     - Week (Weekly Digest)
#     - Research Question (RQ_DB)
#     - Evidence Papers / Evidence Events (relations)
#   And stores:
#     - Category (rich_text)
#     - Open Questions (UPDATED FINAL TEXT, Japanese)
#     - Update Summary (REASON / WHY)
#     - Confidence (0..1)
#     - Status = Proposed
#
# Inputs:
#   - rq_updates_linked (from Cell 13)
#   - notion_request_json() (from Cell 03)
#   - RESOLVED_DB["WEEKLY_RQ_UPDATE_DB"]["database_id"] (resolved in Cell 03)
#
# Outputs:
#   - created_rq_update_ids (list[str])
#   - write_errors (list[str])
#
# Notes:
#   - Fail-soft: continues on errors.
#

from __future__ import annotations
from typing import Dict, Any, List

# ---- Resolve target database_id safely ----
if "RESOLVED_DB" not in globals():
    raise NameError("RESOLVED_DB is not defined. Run Cell 03 first.")

if "WEEKLY_RQ_UPDATE_DB" not in RESOLVED_DB:
    raise NameError('RESOLVED_DB["WEEKLY_RQ_UPDATE_DB"] is missing. Check Cell 03 resolution keys.')

WEEKLY_RQ_UPDATE_DB_ID = RESOLVED_DB["WEEKLY_RQ_UPDATE_DB"]["database_id"]
if not WEEKLY_RQ_UPDATE_DB_ID:
    raise ValueError("WEEKLY_RQ_UPDATE_DB_ID is empty. Check env + Cell 03 resolution.")

created_rq_update_ids: List[str] = []
write_errors: List[str] = []

print("[Cell 14] Writing per-category RQ update pages to WEEKLY_RQ_UPDATE_DB...\n")
print(f"  target database_id = {WEEKLY_RQ_UPDATE_DB_ID}")

# ---- Property names (edit here if your Notion names differ) ----
PROP_TITLE = "Name"
PROP_WEEK = "Week"
PROP_THEMES = "Related Theme(s)"          # optional
PROP_RQ = "Research Question"
PROP_UPDATE_SUMMARY = "Update Summary"
PROP_OPEN_QUESTIONS = "Open Questions"
PROP_CONFIDENCE = "Confidence"
PROP_STATUS = "Status"
PROP_CATEGORY = "Category"
PROP_EVID_EVENTS = "Evidence Events"
PROP_EVID_PAPERS = "Evidence Papers"

def _rt(text: str) -> List[Dict[str, Any]]:
    text = (text or "").strip()
    if not text:
        return []
    return [{"type": "text", "text": {"content": text}}]

for i, item in enumerate(rq_updates_linked, 1):
    try:
        # ---- Build properties payload ----
        properties: Dict[str, Any] = {
            # Title
            PROP_TITLE: {"title": _rt(item.get("name_title", ""))},

            # Category (TEXT)
            PROP_CATEGORY: {"rich_text": _rt(item.get("category_text", "NoChange"))},

            # Open Questions = UPDATED FINAL TEXT (Japanese)
            PROP_OPEN_QUESTIONS: {"rich_text": _rt(item.get("open_q_text", ""))},

            # Update Summary = reason
            PROP_UPDATE_SUMMARY: {"rich_text": _rt(item.get("reason_text", ""))},

            # Confidence
            PROP_CONFIDENCE: {"number": float(item.get("confidence_norm", 0.0))},

            # Status default
            PROP_STATUS: {"select": {"name": "Proposed"}},

            # Relations
            PROP_RQ: {"relation": item.get("rq_rel", []) or []},
            PROP_EVID_PAPERS: {"relation": item.get("evidence_papers_rel", []) or []},
            PROP_EVID_EVENTS: {"relation": item.get("evidence_events_rel", []) or []},
        }

        # Optional: Week relation (if present in schema + we have it)
        if item.get("week_rel"):
            properties[PROP_WEEK] = {"relation": item["week_rel"]}

        # Optional: Related Theme(s) relation (if you populate later)
        if item.get("themes_rel"):
            properties[PROP_THEMES] = {"relation": item["themes_rel"]}

        payload = {
            "parent": {"database_id": WEEKLY_RQ_UPDATE_DB_ID},
            "properties": properties,
        }

        # ---- Create page ----
        resp = notion_request_json("POST", "pages", json_body=payload)
        page_id = resp.get("id")
        
        # ---- Verify by reading back the page ----
        page = notion_request_json("GET", f"pages/{page_id}")
        
        def _prop(page, name):
            return (page.get("properties", {}) or {}).get(name, {}) or {}
        
        evid_p = _prop(page, "Evidence Papers")
        evid_e = _prop(page, "Evidence Events")
        
        print("[VERIFY] created_page_id =", page_id)
        print("[VERIFY] Evidence Papers type =", evid_p.get("type"))
        print("[VERIFY] Evidence Papers relation len =", len((evid_p.get("relation") or [])))
        print("[VERIFY] Evidence Events relation len =", len((evid_e.get("relation") or [])))


        created_rq_update_ids.append(page_id)

        np = len(item.get("evidence_papers_rel", []) or [])
        ne = len(item.get("evidence_events_rel", []) or [])
        print(f"  ✓ [{i}] Created: {item.get('name_title','')}")
        print(f"      category={item.get('category_text')} | evidence: {np} papers, {ne} events | page_id={page_id}")

        # Also attach created id back to the item (handy for Cell 15/16)
        item["created_page_id"] = page_id

    except Exception as e:
        msg = f"[{i}] Failed to create update page ({item.get('name_title','')}): {str(e)}"
        write_errors.append(msg)
        print(f"  ✗ {msg}")

print("\n[Cell 14] Write complete.")
print(f"  Created pages : {len(created_rq_update_ids)}")
print(f"  Write errors  : {len(write_errors)}")


[Cell 14] Writing per-category RQ update pages to WEEKLY_RQ_UPDATE_DB...

  target database_id = 2ff8e0e4-d162-8039-b05c-ed10ea919ca5
[VERIFY] created_page_id = 3008e0e4-d162-8177-bfa9-fe45f8f801ed
[VERIFY] Evidence Papers type = relation
[VERIFY] Evidence Papers relation len = 1
[VERIFY] Evidence Events relation len = 1
  ✓ [1] Created: [v1.1] スタートアップ投資バブルを長期的なイノベーションの軌道に変換するための神話や物語構造と、CVC戦略の役割は何か？: Rationale (2026-02-02–2026-02-08)
      category=Rationale | evidence: 1 papers, 1 events | page_id=3008e0e4-d162-8177-bfa9-fe45f8f801ed
[VERIFY] created_page_id = 3008e0e4-d162-8191-8456-ea8aee7f7d66
[VERIFY] Evidence Papers type = relation
[VERIFY] Evidence Papers relation len = 1
[VERIFY] Evidence Events relation len = 0
  ✓ [2] Created: [v1.1] 外国系VCとコーポレートVCは、新興市場におけるスタートアップの成長とイノベーション成果にどのように異なる影響を与えるか？: Rationale (2026-02-02–2026-02-08)
      category=Rationale | evidence: 1 papers, 0 events | page_id=3008e0e4-d162-8191-8456-ea8aee7f7d66
[VERIFY] created_page_id = 3008e0e4-d162-81f1

In [40]:
# ============================================================
# Cell 15 — Update weekly digest RQ Updates relation
# ============================================================
# Overview:
#   Patch the most recent WEEKLY_DIGESTS page (loaded in Cell 07) to update only
#   the "RQ Updates" relation, linking all newly created WEEKLY_RQ_UPDATE_DB pages.
#   Other digest fields must remain unchanged.
#
# Inputs / Outputs:
#   Inputs:
#     - notion_request_json() from Cell 03
#     - weekly_digest_id, digest_title, existing_rq_updates from Cell 07
#     - created_rq_update_ids from Cell 14 (list[str])  # preferred
#       (fallback: rq_updates_linked items may include created_page_id if you store it there)
#   Outputs:
#     - updated_weekly_digest (dict): Notion API response (PATCH /pages/{id})
#     - final_rq_update_relation_ids (list[str]): merged relation ids written to the digest
#
# Notes:
#   - Uses PATCH /v1/pages/{page_id} and sets the full relation list.
#   - Default behavior is to MERGE with existing links (dedupe, preserve order as much as possible).
#   - If you want to REPLACE existing links, set MERGE_WITH_EXISTING=False.
#

from __future__ import annotations
from typing import Any, Dict, List, Set

MERGE_WITH_EXISTING = True
RQ_UPDATES_PROP = "RQ Updates"

def _clean_ids(xs: Any) -> List[str]:
    if not xs:
        return []
    if isinstance(xs, list):
        out: List[str] = []
        for x in xs:
            if isinstance(x, str) and x.strip():
                out.append(x.strip())
            elif isinstance(x, dict) and x.get("id"):
                out.append(str(x["id"]).strip())
        return [i for i in out if i]
    if isinstance(xs, str) and xs.strip():
        return [xs.strip()]
    return []

def _dedupe_preserve(xs: List[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in xs:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _rel(ids: List[str]) -> List[Dict[str, str]]:
    return [{"id": i} for i in ids if isinstance(i, str) and i.strip()]

print("[Cell 15] Updating weekly digest RQ Updates relation...\n")

# --- Prefer created_rq_update_ids from Cell 14 ---
new_ids = _clean_ids(globals().get("created_rq_update_ids", []))

# --- Fallback: if you stored per-item created_page_id somewhere ---
if not new_ids:
    try:
        # rq_updates_linked is from Cell 13; if you later added created_page_id to those dicts, this picks them up
        fallback_items = globals().get("rq_updates_linked", []) or []
        new_ids = _clean_ids([it.get("created_page_id") for it in fallback_items if isinstance(it, dict)])
    except Exception:
        new_ids = []

existing_ids = _clean_ids(globals().get("existing_rq_updates", []))

print(f"  Newly created RQ update pages: {len(new_ids)}")
print(f"  Existing RQ update links in digest: {len(existing_ids)}")

if not new_ids:
    print("\n  No new RQ updates to link. Weekly digest unchanged.")
    updated_weekly_digest = None
    final_rq_update_relation_ids = _dedupe_preserve(existing_ids)
else:
    if MERGE_WITH_EXISTING:
        final_ids = _dedupe_preserve(existing_ids + new_ids)
    else:
        final_ids = _dedupe_preserve(new_ids)

    print(f"  Total RQ update links to write: {len(final_ids)} (merge={MERGE_WITH_EXISTING})")

    patch_payload = {
        "properties": {
            RQ_UPDATES_PROP: {"relation": _rel(final_ids)}
        }
    }

    try:
        updated_weekly_digest = notion_request_json("PATCH", f"pages/{weekly_digest_id}", json_body=patch_payload)

        print("\n  ✓ Weekly digest updated successfully")
        print(f"    Digest ID : {weekly_digest_id}")
        print(f"    Title     : {digest_title}")
        print(f"    Added     : {len(new_ids)} new RQ update relations")
        print(f"    Total     : {len(final_ids)}")

        final_rq_update_relation_ids = final_ids

    except Exception as e:
        msg = f"Failed to update weekly digest RQ Updates relation: {str(e)}"
        print(f"\n  ✗ {msg}")
        raise

print("\n[Cell 15] Done.")


[Cell 15] Updating weekly digest RQ Updates relation...

  Newly created RQ update pages: 20
  Existing RQ update links in digest: 0
  Total RQ update links to write: 20 (merge=True)

  ✓ Weekly digest updated successfully
    Digest ID : 2ff8e0e4-d162-8165-9a05-e4061df44fb5
    Title     : Weekly Events Digest: 2026-02-02–2026-02-08
    Added     : 20 new RQ update relations
    Total     : 20

[Cell 15] Done.


In [41]:
# ============================================================
# Cell 16 — Generate execution summary and validation report
# ============================================================
# Overview:
#   Produce a final execution summary for 042_weekly_rq_status.
#   This report is CATEGORY-AWARE and PAGE-BASED (not per-RQ),
#   reflecting the current design where each WEEKLY_RQ_UPDATE_DB page
#   corresponds to exactly ONE category item:
#
#     - RQ
#     - LinkedPaper
#     - Rationale
#     - Approach
#     - Gap
#
#   Display contract:
#     - Open Questions  : stores UPDATED text (after-change, Japanese)
#     - Update Summary  : stores the REASON / WHY (English or Japanese)
#
# Inputs / Outputs:
#   Inputs:
#     - weekly_digest_id, digest_title, digest_week_start, digest_week_end
#     - rq_records
#     - paper_records, event_records
#     - rq_updates_norm        (from Cell 12)
#     - created_rq_update_ids  (from Cell 14)
#     - write_errors           (from Cell 14)
#   Outputs:
#     - execution_summary (dict)
#
# Notes:
#   - This cell does NOT modify Notion.
#   - It validates semantic correctness, not just counts.
#

from collections import Counter

# ---- Defensive defaults (in case upstream cells changed variable names) ----
created_rq_update_ids = globals().get("created_rq_update_ids", []) or []

# write_errors may not exist in the new design
write_errors = globals().get("write_errors", []) or []

print("=" * 80)
print("NOTEBOOK EXECUTION SUMMARY: 042_weekly_rq_status")
print("=" * 80)
print()

# ------------------------------------------------------------------
# 1. Weekly digest context
# ------------------------------------------------------------------
print("1. WEEKLY DIGEST CONTEXT")
print("-" * 80)
print(f"  Digest ID   : {weekly_digest_id}")
print(f"  Title       : {digest_title}")
print(f"  Week Range  : {digest_week_start} → {digest_week_end}")
print()

# ------------------------------------------------------------------
# 2. Input data summary
# ------------------------------------------------------------------
print("2. INPUT DATA SUMMARY")
print("-" * 80)
print(f"  Target RQs (Priority=High): {len(rq_records)}")
print(f"  Evidence Papers            : {len(paper_records)}")
print(f"  Weekly Events              : {len(event_records)}")
print()

# ------------------------------------------------------------------
# 3. Generated update pages (category-based)
# ------------------------------------------------------------------
print("3. GENERATED RQ UPDATE PAGES (BY CATEGORY)")
print("-" * 80)

total_items = len(rq_updates_norm)
created_items = len(created_rq_update_ids)
failed_items = len(write_errors)

categories = [u["category_text"] for u in rq_updates_norm]
cat_counter = Counter(categories)

print(f"  Total update pages generated : {total_items}")
print(f"  Successfully written         : {created_items}")
print(f"  Write failures               : {failed_items}")
print()

print("  Breakdown by Category:")
for cat, cnt in cat_counter.items():
    print(f"    - {cat:12s}: {cnt}")
print()

# ------------------------------------------------------------------
# 4. Content quality checks (semantic)
# ------------------------------------------------------------------
print("4. CONTENT QUALITY CHECKS")
print("-" * 80)

issues = []

def _missing_open_questions(u):
    return not (u.get("open_questions_norm") or [])

def _missing_reason(u):
    return not (u.get("update_summary") or "").strip()

# Category-specific expectations
for u in rq_updates_norm:
    cat = u["category_text"]

    # All categories except LinkedPaper require Open Questions text
    if cat in {"RQ", "Rationale", "Approach", "Gap"}:
        if _missing_open_questions(u):
            issues.append(f"[{cat}] Missing updated text (Open Questions) for RQ='{u.get('rq_title')}'")

    # All categories require a reason
    if _missing_reason(u):
        issues.append(f"[{cat}] Missing rationale / reason text for RQ='{u.get('rq_title')}'")

    # LinkedPaper must actually add papers
    if cat == "LinkedPaper" and not u.get("linked_paper_additions"):
        issues.append(f"[LinkedPaper] No papers added for RQ='{u.get('rq_title')}'")

print(f"  Detected issues: {len(issues)}")
if issues:
    for i, msg in enumerate(issues, 1):
        print(f"    {i}. {msg}")
else:
    print("  ✓ No semantic issues detected")
print()

# ------------------------------------------------------------------
# 5. Confidence overview (informational)
# ------------------------------------------------------------------
print("5. CONFIDENCE OVERVIEW")
print("-" * 80)

conf_vals = [u["confidence_norm"] for u in rq_updates_norm if u.get("confidence_norm") is not None]
if conf_vals:
    print(f"  Avg confidence : {sum(conf_vals)/len(conf_vals):.2f}")
    print(f"  Min / Max      : {min(conf_vals):.2f} / {max(conf_vals):.2f}")
else:
    print("  No confidence values present")
print()

# ------------------------------------------------------------------
# 6. Overall status
# ------------------------------------------------------------------
print("6. OVERALL EXECUTION STATUS")
print("-" * 80)

if not write_errors and not issues:
    status = "SUCCESS"
    print("  ✓ SUCCESS: All pages written and content looks consistent")
elif not write_errors:
    status = "PARTIAL_SUCCESS"
    print("  ! PARTIAL SUCCESS: Pages written but content issues detected")
else:
    status = "FAILURE"
    print("  ✗ FAILURE: Write errors occurred")

print(f"  Status: {status}")
print()

# ------------------------------------------------------------------
# 7. Next steps
# ------------------------------------------------------------------
print("7. NEXT STEPS")
print("-" * 80)

print("  1. Review WEEKLY_RQ_UPDATE_DB pages grouped by Category")
print("  2. For Rationale / Approach / Gap:")
print("     - Confirm Open Questions text is the FINAL wording to merge into RQ_DB")
print("  3. Approve or reject each page via Status field in Notion")
print("  4. Apply accepted patches back to RQ_DB (future automation)")
print()

print("=" * 80)
print("END OF EXECUTION SUMMARY")
print("=" * 80)

# ------------------------------------------------------------------
# Export structured summary
# ------------------------------------------------------------------
execution_summary = {
    "status": status,
    "total_update_pages": total_items,
    "pages_written": created_items,
    "write_errors": write_errors,
    "category_breakdown": dict(cat_counter),
    "content_issues": issues,
    "digest_id": weekly_digest_id,
}

print("\nExecution summary stored in `execution_summary`.")


NOTEBOOK EXECUTION SUMMARY: 042_weekly_rq_status

1. WEEKLY DIGEST CONTEXT
--------------------------------------------------------------------------------
  Digest ID   : 2ff8e0e4-d162-8165-9a05-e4061df44fb5
  Title       : Weekly Events Digest: 2026-02-02–2026-02-08
  Week Range  : 2026-02-02 → 2026-02-08

2. INPUT DATA SUMMARY
--------------------------------------------------------------------------------
  Target RQs (Priority=High): 8
  Evidence Papers            : 7
  Weekly Events              : 54

3. GENERATED RQ UPDATE PAGES (BY CATEGORY)
--------------------------------------------------------------------------------
  Total update pages generated : 20
  Successfully written         : 20
  Write failures               : 0

  Breakdown by Category:
    - Rationale   : 8
    - Approach    : 6
    - Gap         : 6

4. CONTENT QUALITY CHECKS
--------------------------------------------------------------------------------
  Detected issues: 0
  ✓ No semantic issues detected

5.